In [1]:
# -*- coding: utf-8 -*-
"""MAPPO — Single Cell for Google Colab"""
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta
from torch.optim import Adam
from collections import deque
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION & HYPERPARAMETERS
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ZINB_H1_PARAMS = [{"phi": 0.25, "p": 0.48, "n": 15}]
ZINB_H2_PARAMS = [{"phi": 0.25, "p": 0.48, "n": 15}]

def _zinb_mean(params: dict) -> float:
    phi, p, n = params["phi"], params["p"], params["n"]
    return (1.0 - phi) * n * (1.0 - p) / p

def sample_zinb(phi: float, p: float, n: int) -> int:
    if np.random.random() < phi:
        return 0
    return int(np.random.negative_binomial(n, p))

def sample_hospital_demand(zinb_params: list, mult: float = 1.0) -> int:
    total = sum(sample_zinb(p["phi"], p["p"], p["n"]) for p in zinb_params)
    return max(0, int(total * mult))

ENV_CONFIG = {
    "max_capacity": 100,
    "shelf_life": 5,
    "order_cost": 1.0,
    "transshipment_cost": 1.0,
    "transshipment_fixed_cost": 5.0,
    "shortage_cost": 25.0,
    "wastage_cost": 15.0,
    "inventory_cost": 0.5,
    "episode_length": 60,
    "order_cap": 70,
    "trans_cap": 50,
    "transshipment_deadband": 0.1,
    "disruption_threshold": 1.2,
    "disruption_prob": 0.02,
    "disruption_min_len": 3,
    "disruption_max_len": 8,
    "disruption_mult": 1.4,
}

ORDER_SCHEDULE_H1 = {1, 4, 6}
ORDER_SCHEDULE_H2 = {0, 3, 5}

STATE_DIM = 26
PROCUREMENT_ACTION_DIM = 2
LOGISTICS_ACTION_DIM = 1
STATE_SCALE = 70.0
REWARD_SCALE = 100.0

LR = 3e-4
GAMMA = 0.99
EPSILON = 0.2
EPOCHS = 4
GAE_LAMBDA = 0.95
UPDATE_EPISODES = 4
ENTROPY_COEF = 0.05
ENTROPY_DECAY = 0.999
MIN_ENTROPY = 0.005
TOTAL_EPISODES = 2000

OUTPUT_PREFIX = "mappo"

# =============================================================================
# ENVIRONMENT
# =============================================================================
class PlateletSupplyChainEnv:
    def __init__(self):
        for k, v in ENV_CONFIG.items():
            setattr(self, k, v)
        self.num_hospitals = 2
        self.demand_mean_h1 = sum(_zinb_mean(p) for p in ZINB_H1_PARAMS)
        self.demand_mean_h2 = sum(_zinb_mean(p) for p in ZINB_H2_PARAMS)
        self.reset()

    def _is_order_day_h1(self):
        return (self.day % 7) in ORDER_SCHEDULE_H1

    def _is_order_day_h2(self):
        return (self.day % 7) in ORDER_SCHEDULE_H2

    def _compute_disruption_flag(self, history):
        if len(history) < 5:
            return 0.0
        ma2 = np.mean(list(history)[-2:])
        ma30 = np.mean(list(history)[-30:])
        return 1.0 if (ma30 > 0 and ma2 > self.disruption_threshold * ma30) else 0.0

    def _get_observation(self):
        obs = np.zeros((STATE_DIM,), dtype=np.float32)
        idx = 0
        for h in range(self.num_hospitals):
            start = h * self.shelf_life
            for age in range(self.shelf_life):
                obs[idx] = self.state[start + age]
                idx += 1

        inv_h1 = np.sum(self.state[0:self.shelf_life])
        inv_h2 = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        obs[10] = self.day / self.episode_length
        obs[11] = self._compute_disruption_flag(self.demand_history_h1)
        obs[12] = self._compute_disruption_flag(self.demand_history_h2)
        obs[13] = inv_h1 / self.max_capacity
        obs[14] = inv_h2 / self.max_capacity
        obs[15] = 1.0 if self._is_order_day_h1() else 0.0
        obs[16] = 1.0 if self._is_order_day_h2() else 0.0
        obs[17] = (inv_h1 - inv_h2) / self.max_capacity
        obs[18] = self.crisis_counter_h1 / self.disruption_max_len
        obs[19] = self.crisis_counter_h2 / self.disruption_max_len

        d1 = np.mean(self.demand_history_h1[-7:]) if self.demand_history_h1 else self.demand_mean_h1
        d2 = np.mean(self.demand_history_h2[-7:]) if self.demand_history_h2 else self.demand_mean_h2
        obs[20] = min(inv_h1 / (d1 * 7 + 1e-8), 3.0)
        obs[21] = min(inv_h2 / (d2 * 7 + 1e-8), 3.0)

        if self.demand_history_h1:
            td1 = sum(self.demand_history_h1[-30:])
            obs[22] = sum(self.shortage_history_h1) / (td1 + 1e-8)
            obs[24] = sum(self.wastage_history_h1) / (td1 + 1e-8)
        if self.demand_history_h2:
            td2 = sum(self.demand_history_h2[-30:])
            obs[23] = sum(self.shortage_history_h2) / (td2 + 1e-8)
            obs[25] = sum(self.wastage_history_h2) / (td2 + 1e-8)
        return obs.copy()

    def reset(self):
        self.state = np.zeros((self.num_hospitals * self.shelf_life,), dtype=np.float32)
        self.day = 0
        self.demand_history_h1, self.demand_history_h2 = [], []
        self.shortage_history_h1, self.shortage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.wastage_history_h1, self.wastage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.disruption_counter_h1 = self.disruption_counter_h2 = 0
        self.crisis_counter_h1 = self.crisis_counter_h2 = 0
        return self._get_observation()

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)

        is_ord_h1 = self._is_order_day_h1()
        is_ord_h2 = self._is_order_day_h2()

        order_h1 = action[0] * self.order_cap if is_ord_h1 else 0.0
        order_h2 = action[1] * self.order_cap if is_ord_h2 else 0.0

        a_t = action[2] * 2.0 - 1.0
        tau = self.transshipment_deadband

        trans_h1_to_h2 = trans_h2_to_h1 = 0.0
        if a_t > tau:
            trans_h1_to_h2 = ((a_t - tau) / (1.0 - tau)) * self.trans_cap
        elif a_t < -tau:
            trans_h2_to_h1 = ((abs(a_t) - tau) / (1.0 - tau)) * self.trans_cap

        inv_h1_before = np.sum(self.state[0:self.shelf_life])
        inv_h2_before = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        actual_order_h1 = self._process_order(0, min(order_h1, max(0.0, self.max_capacity - inv_h1_before)))
        actual_order_h2 = self._process_order(1, min(order_h2, max(0.0, self.max_capacity - inv_h2_before)))

        actual_trans_h1_to_h2 = self._transfer_blood(0, 1, trans_h1_to_h2)
        actual_trans_h2_to_h1 = self._transfer_blood(1, 0, trans_h2_to_h1)

        if self.day >= 5:
            if self.disruption_counter_h1 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h1 = random.randint(self.disruption_min_len, self.disruption_max_len)
            if self.disruption_counter_h2 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h2 = random.randint(self.disruption_min_len, self.disruption_max_len)

        mult_h1 = self.disruption_mult if self.disruption_counter_h1 > 0 else 1.0
        mult_h2 = self.disruption_mult if self.disruption_counter_h2 > 0 else 1.0

        if self.disruption_counter_h1 > 0:
            self.disruption_counter_h1 -= 1
        if self.disruption_counter_h2 > 0:
            self.disruption_counter_h2 -= 1

        demand_h1 = sample_hospital_demand(ZINB_H1_PARAMS, mult=mult_h1)
        demand_h2 = sample_hospital_demand(ZINB_H2_PARAMS, mult=mult_h2)

        self.demand_history_h1.append(demand_h1)
        self.demand_history_h2.append(demand_h2)

        self.crisis_counter_h1 = min(self.crisis_counter_h1 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h1) > 0.5 else 0
        self.crisis_counter_h2 = min(self.crisis_counter_h2 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h2) > 0.5 else 0

        wastage_h1, shortage_h1, _ = self._fulfill_demand(0, demand_h1)
        wastage_h2, shortage_h2, _ = self._fulfill_demand(1, demand_h2)

        self.shortage_history_h1.append(shortage_h1)
        self.shortage_history_h2.append(shortage_h2)
        self.wastage_history_h1.append(wastage_h1)
        self.wastage_history_h2.append(wastage_h2)

        self._age_blood()

        inv_h1_after = np.sum(self.state[0:self.shelf_life])
        inv_h2_after = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        inventory_holding_cost = (inv_h1_before + inv_h2_before) * self.inventory_cost
        self.day += 1

        total_transferred = actual_trans_h1_to_h2 + actual_trans_h2_to_h1
        transshipment_cost = total_transferred * self.transshipment_cost + (self.transshipment_fixed_cost if total_transferred >= 1.0 else 0.0)
        wastage_cost = (wastage_h1 + wastage_h2) * self.wastage_cost
        shortage_cost = (shortage_h1 + shortage_h2) * self.shortage_cost
        order_cost = (actual_order_h1 + actual_order_h2) * self.order_cost

        total_cost = wastage_cost + shortage_cost + transshipment_cost + order_cost + inventory_holding_cost
        shared_reward = -total_cost / REWARD_SCALE

        info = {
            'order_mask': [1.0 if is_ord_h1 else 0.0, 1.0 if is_ord_h2 else 0.0],
            'demand_h1': demand_h1, 'demand_h2': demand_h2,
            'inventory_h1': inv_h1_after, 'inventory_h2': inv_h2_after,
            'order_h1': order_h1, 'order_h2': order_h2,
            'actual_order_h1': actual_order_h1, 'actual_order_h2': actual_order_h2,
            'transferred_h1_to_h2': actual_trans_h1_to_h2, 'transferred_h2_to_h1': actual_trans_h2_to_h1,
            'shortage_h1': shortage_h1, 'shortage_h2': shortage_h2,
            'wastage_h1': wastage_h1, 'wastage_h2': wastage_h2,
            'total_cost': total_cost, 'wastage_cost': wastage_cost,
            'shortage_cost': shortage_cost, 'transshipment_cost': transshipment_cost,
            'order_cost': order_cost, 'inventory_holding_cost': inventory_holding_cost,
            'disruption_active_h1': mult_h1 > 1.0, 'disruption_active_h2': mult_h2 > 1.0,
            'crisis_counter_h1': self.crisis_counter_h1, 'crisis_counter_h2': self.crisis_counter_h2,
            'inv_h1_before': inv_h1_before, 'inv_h2_before': inv_h2_before,
        }

        done = self.day >= self.episode_length
        return self._get_observation(), shared_reward, done, info

    def _process_order(self, hospital, amount):
        if amount <= 0:
            return 0.0
        start = hospital * self.shelf_life
        accepted = max(0.0, min(amount, self.max_capacity - np.sum(self.state[start:start + self.shelf_life])))
        self.state[start] += accepted
        return accepted

    def _transfer_blood(self, from_h, to_h, amount):
        if amount <= 0:
            return 0.0
        f_s, f_e = from_h * self.shelf_life, from_h * self.shelf_life + self.shelf_life
        t_s, t_e = to_h * self.shelf_life, to_h * self.shelf_life + self.shelf_life

        transferrable = max(0.0, min(amount, np.sum(self.state[f_s:f_e]), self.max_capacity - np.sum(self.state[t_s:t_e])))
        transferred = 0.0
        for i in range(f_e - 1, f_s - 1, -1):
            if self.state[i] > 0:
                to_transfer = min(self.state[i], transferrable - transferred)
                self.state[i] -= to_transfer
                self.state[t_s + (i - f_s)] += to_transfer
                transferred += to_transfer
                if transferred >= transferrable:
                    break
        return transferred

    def _fulfill_demand(self, hospital, demand):
        s_idx, e_idx = hospital * self.shelf_life, hospital * self.shelf_life + self.shelf_life
        unfulfilled = demand
        for i in range(e_idx - 1, s_idx - 1, -1):
            if self.state[i] > 0 and unfulfilled > 0:
                used = min(self.state[i], unfulfilled)
                self.state[i] -= used
                unfulfilled -= used
            if unfulfilled <= 1e-5:
                break
        wastage = self.state[e_idx - 1]
        self.state[e_idx - 1] = 0
        return wastage, unfulfilled, demand - unfulfilled

    def _age_blood(self):
        for h in range(self.num_hospitals):
            s_idx, e_idx = h * self.shelf_life, h * self.shelf_life + self.shelf_life
            self.state[s_idx + 1:e_idx] = self.state[s_idx:e_idx - 1]
            self.state[s_idx] = 0


# =============================================================================
# RL ALGORITHMS (MAPPO)
# =============================================================================
class RunningMeanStd:
    def __init__(self, epsilon=1e-4, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = epsilon

    def update(self, x):
        batch_mean = np.mean(x)
        batch_var = np.var(x)
        batch_count = x.shape[0]
        self.update_from_moments(batch_mean, batch_var, batch_count)

    def update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        tot_count = self.count + batch_count
        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot_count
        new_var = M2 / tot_count
        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def normalize(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


class Actor(nn.Module):
    def __init__(self, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(STATE_DIM, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())
        self.alpha_head = nn.Linear(256, action_dim)
        self.beta_head = nn.Linear(256, action_dim)

    def forward(self, state):
        feat = self.net(state)
        alpha = torch.clamp(F.softplus(self.alpha_head(feat)) + 1.0, 1.0, 100.0)
        beta = torch.clamp(F.softplus(self.beta_head(feat)) + 1.0, 1.0, 100.0)
        return Beta(alpha, beta)

    def get_action(self, state):
        with torch.no_grad():
            dist = self.forward(state)
            action = dist.sample()
            safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
            return safe_act.squeeze(0).cpu().numpy(), dist.log_prob(safe_act).squeeze(0).cpu().numpy()

    def evaluate(self, state, action):
        dist = self.forward(state)
        safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
        return dist.log_prob(safe_act), dist.entropy()


class MultiAgentMAPPO:
    def __init__(self):
        self.proc_actor = Actor(PROCUREMENT_ACTION_DIM).to(DEVICE)
        self.log_actor = Actor(LOGISTICS_ACTION_DIM).to(DEVICE)
        self.critic = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        ).to(DEVICE)

        self.opt_p = Adam(self.proc_actor.parameters(), lr=LR)
        self.opt_l = Adam(self.log_actor.parameters(), lr=LR)
        self.opt_c = Adam(self.critic.parameters(), lr=LR)
        self.entropy_coef = ENTROPY_COEF
        self.rms = RunningMeanStd()

    def _norm(self, st):
        st = st.clone()
        st[:, :10] /= STATE_SCALE
        return st

    def get_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        ap, lp = self.proc_actor.get_action(st)
        al, ll = self.log_actor.get_action(st)
        return np.concatenate([ap, al]), ap, al, lp, ll

    def get_deterministic_joint_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        with torch.no_grad():
            act_proc = self.proc_actor.forward(st).mean.squeeze(0).cpu().numpy()
            act_log = self.log_actor.forward(st).mean.squeeze(0).cpu().numpy()
        return np.concatenate([act_proc, act_log], axis=-1)

    def save(self, path):
        torch.save({
            'proc_actor': self.proc_actor.state_dict(),
            'log_actor': self.log_actor.state_dict(),
            'critic': self.critic.state_dict()
        }, path)

    def load(self, path):
        ck = torch.load(path, map_location=DEVICE, weights_only=True)
        self.proc_actor.load_state_dict(ck['proc_actor'])
        self.log_actor.load_state_dict(ck['log_actor'])
        self.critic.load_state_dict(ck['critic'])

    def update(self, s, ap, al, lp, ll, r, ns, d, masks):
        s = self._norm(torch.FloatTensor(np.array(s)).to(DEVICE))
        ns = self._norm(torch.FloatTensor(np.array(ns)).to(DEVICE))
        ap = torch.FloatTensor(np.array(ap)).to(DEVICE)
        al = torch.FloatTensor(np.array(al)).to(DEVICE)
        lp = torch.FloatTensor(np.array(lp)).to(DEVICE)
        ll = torch.FloatTensor(np.array(ll)).to(DEVICE)

        r_np = np.array(r)
        self.rms.update(r_np)
        r_norm = torch.FloatTensor(self.rms.normalize(r_np)).to(DEVICE)

        d = torch.FloatTensor(np.array(d)).to(DEVICE)
        masks = torch.FloatTensor(np.array(masks)).to(DEVICE)

        with torch.no_grad():
            vals = self.critic(s).squeeze()
            n_vals = self.critic(ns).squeeze()

        adv = torch.zeros_like(r_norm).to(DEVICE)
        gae = 0
        for t in reversed(range(len(r_norm))):
            delta = r_norm[t] + GAMMA * n_vals[t] * (1 - d[t]) - vals[t]
            gae = delta + GAMMA * GAE_LAMBDA * (1 - d[t]) * gae
            adv[t] = gae

        returns = adv + vals
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(EPOCHS):
            new_logp_p, ent_p = self.proc_actor.evaluate(s, ap)
            ratio_p = torch.exp(new_logp_p - lp.detach())
            surr1_p = ratio_p * adv.unsqueeze(1)
            surr2_p = torch.clamp(ratio_p, 1-EPSILON, 1+EPSILON) * adv.unsqueeze(1)
            loss_p = -torch.min(surr1_p, surr2_p) * masks
            loss_p = loss_p.sum() / (masks.sum() + 1e-8)
            loss_p -= self.entropy_coef * (ent_p * masks).sum() / (masks.sum() + 1e-8)

            self.opt_p.zero_grad()
            loss_p.backward()
            nn.utils.clip_grad_norm_(self.proc_actor.parameters(), 0.5)
            self.opt_p.step()

            new_logp_l, ent_l = self.log_actor.evaluate(s, al)
            ratio_l = torch.exp(new_logp_l - ll.detach())
            surr1_l = ratio_l * adv.unsqueeze(1)
            surr2_l = torch.clamp(ratio_l, 1-EPSILON, 1+EPSILON) * adv.unsqueeze(1)
            loss_l = -torch.min(surr1_l, surr2_l).mean() - self.entropy_coef * ent_l.mean()

            self.opt_l.zero_grad()
            loss_l.backward()
            nn.utils.clip_grad_norm_(self.log_actor.parameters(), 0.5)
            self.opt_l.step()

            v_pred = self.critic(s).squeeze()
            loss_c = F.mse_loss(v_pred, returns)
            self.opt_c.zero_grad()
            loss_c.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
            self.opt_c.step()

        self.entropy_coef = max(MIN_ENTROPY, self.entropy_coef * ENTROPY_DECAY)


# =============================================================================
# TRAINING FUNCTION
# =============================================================================
def train(train_seed=42):
    print(f"\n{'='*80}")
    print(f"🚀 TRAINING SEED {train_seed}")
    print(f"{'='*80}")

    random.seed(train_seed)
    np.random.seed(train_seed)
    torch.manual_seed(train_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(train_seed)

    env = PlateletSupplyChainEnv()
    mappo = MultiAgentMAPPO()

    buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []
    model_path = None

    for ep in tqdm(range(TOTAL_EPISODES), desc=f"Train Seed {train_seed}"):
        s = env.reset()
        ep_r = 0
        for step in range(env.episode_length):
            j_act, ap, al, lp, ll = mappo.get_action(s)
            ns, r, d, info = env.step(j_act)

            buf_s.append(s); buf_ap.append(ap); buf_al.append(al)
            buf_lp.append(lp); buf_ll.append(ll); buf_r.append(r)
            buf_ns.append(ns); buf_d.append(d); buf_m.append(info['order_mask'])

            s = ns
            ep_r += r
            if d:
                break

        rewards_log.append(ep_r)

        if (ep + 1) % UPDATE_EPISODES == 0:
            mappo.update(buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], [], [], []

        if ep >= 100:
            avg_r = np.mean(rewards_log[-100:])
            if avg_r > best_reward:
                best_reward = avg_r
                model_path = f"{OUTPUT_PREFIX}_seed{train_seed}.pt"
                mappo.save(model_path)

    return model_path, rewards_log


# =============================================================================
# COMBINED LEARNING CURVE
# =============================================================================
def plot_combined_learning_curves(all_rewards):
    print(f"\n{'='*80}")
    print(f"📈 GENERATING COMBINED LEARNING CURVE")
    print(f"{'='*80}")

    plt.figure(figsize=(12, 6))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    for idx, (seed, rewards) in enumerate(all_rewards.items()):
        if len(rewards) >= 50:
            smoothed = np.convolve(rewards, np.ones(50)/50, mode='valid')
            episodes = np.arange(49, len(rewards))
            plt.plot(episodes, smoothed, color=colors[idx % len(colors)],
                     linewidth=2.5, label=f'Model {idx+1} (Seed {seed})')
        plt.plot(rewards, alpha=0.08, color=colors[idx % len(colors)])

    plt.xlabel("Episode", fontsize=13, fontweight='bold')
    plt.ylabel("Episode Reward", fontsize=13, fontweight='bold')
    plt.title("MAPPO Training Curves — Platelet Supply Chain", fontsize=15, fontweight='bold')
    plt.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_PREFIX}_combined_learning_curves.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"💾 Saved: {OUTPUT_PREFIX}_combined_learning_curves.png")


# =============================================================================
# EVALUATION HELPERS
# =============================================================================
H1_COLS = ['Total_Demand', 'Fulfilled_Demand', 'Unfulfilled_Demand', 'Wastage_Units',
           'Shortage_Units', 'Wastage_Rate_Pct', 'Shortage_Rate_Pct', 'Wastage_Cost',
           'Shortage_Cost', 'Holding_Cost', 'Order_Cost', 'Total_Cost']
H2_COLS = H1_COLS
SC_COLS = ['Total_Wastage', 'Total_Shortage', 'Trans_H1_to_H2', 'Trans_H2_to_H1',
           'Trans_Cost_H1_to_H2', 'Trans_Cost_H2_to_H1', 'Network_Total_Cost']


def compute_episode_summary(df, cols):
    s = {}
    for c in cols:
        if 'Cost' in c or 'Units' in c or 'Demand' in c or 'Trans' in c:
            s[c] = df[c].sum()
        else:
            s[c] = df[c].mean()
    return s


def aggregate_seed_summaries(seed_summaries):
    df = pd.DataFrame(seed_summaries)
    result = {}
    for c in df.columns:
        result[f'{c}_Mean'] = df[c].mean()
        result[f'{c}_Std'] = df[c].std()
    return pd.DataFrame([result])


def evaluate_multi_seed(model_path, train_seed, eval_seeds=range(1, 51)):
    print(f"\n{'='*80}")
    print(f"🚀 EVALUATING MODEL (Train Seed {train_seed}) ON {len(eval_seeds)} SEEDS")
    print(f"{'='*80}")

    env = PlateletSupplyChainEnv()
    mappo = MultiAgentMAPPO()
    mappo.load(model_path)
    mappo.proc_actor.eval()
    mappo.log_actor.eval()

    h1_records_all = []
    h2_records_all = []
    sc_records_all = []

    for seed in tqdm(eval_seeds, desc=f"Eval Train-Seed {train_seed}"):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        state = env.reset()
        h1_daily = []
        h2_daily = []
        sc_daily = []

        for day in range(env.episode_length):
            action = mappo.get_deterministic_joint_action(state)
            next_state, reward, done, info = env.step(action)

            d1 = info['demand_h1']
            s1 = info['shortage_h1']
            w1 = info['wastage_h1']
            fulfilled1 = d1 - s1
            inv_h1_before = info['inv_h1_before']
            inv_h2_before = info['inv_h2_before']
            total_inv_before = inv_h1_before + inv_h2_before

            h1_hold = (inv_h1_before / total_inv_before) * info['inventory_holding_cost'] if total_inv_before > 0 else 0.0

            h1_daily.append({
                'Day': day + 1,
                'Total_Demand': d1,
                'Fulfilled_Demand': fulfilled1,
                'Unfulfilled_Demand': s1,
                'Wastage_Units': w1,
                'Shortage_Units': s1,
                'Wastage_Rate_Pct': (w1 / d1 * 100) if d1 > 0 else 0.0,
                'Shortage_Rate_Pct': (s1 / d1 * 100) if d1 > 0 else 0.0,
                'Wastage_Cost': w1 * env.wastage_cost,
                'Shortage_Cost': s1 * env.shortage_cost,
                'Holding_Cost': h1_hold,
                'Order_Cost': info['actual_order_h1'] * env.order_cost,
                'Total_Cost': (w1 * env.wastage_cost) + (s1 * env.shortage_cost) + h1_hold + (info['actual_order_h1'] * env.order_cost),
            })

            d2 = info['demand_h2']
            s2 = info['shortage_h2']
            w2 = info['wastage_h2']
            fulfilled2 = d2 - s2

            h2_hold = (inv_h2_before / total_inv_before) * info['inventory_holding_cost'] if total_inv_before > 0 else 0.0

            h2_daily.append({
                'Day': day + 1,
                'Total_Demand': d2,
                'Fulfilled_Demand': fulfilled2,
                'Unfulfilled_Demand': s2,
                'Wastage_Units': w2,
                'Shortage_Units': s2,
                'Wastage_Rate_Pct': (w2 / d2 * 100) if d2 > 0 else 0.0,
                'Shortage_Rate_Pct': (s2 / d2 * 100) if d2 > 0 else 0.0,
                'Wastage_Cost': w2 * env.wastage_cost,
                'Shortage_Cost': s2 * env.shortage_cost,
                'Holding_Cost': h2_hold,
                'Order_Cost': info['actual_order_h2'] * env.order_cost,
                'Total_Cost': (w2 * env.wastage_cost) + (s2 * env.shortage_cost) + h2_hold + (info['actual_order_h2'] * env.order_cost),
            })

            total_trans = info['transferred_h1_to_h2'] + info['transferred_h2_to_h1']
            trans_cost_total = info['transshipment_cost']

            if total_trans > 0:
                trans_cost_h1_to_h2 = (info['transferred_h1_to_h2'] / total_trans) * trans_cost_total
                trans_cost_h2_to_h1 = (info['transferred_h2_to_h1'] / total_trans) * trans_cost_total
            else:
                trans_cost_h1_to_h2 = 0.0
                trans_cost_h2_to_h1 = 0.0

            sc_daily.append({
                'Day': day + 1,
                'Total_Wastage': w1 + w2,
                'Total_Shortage': s1 + s2,
                'Trans_H1_to_H2': info['transferred_h1_to_h2'],
                'Trans_H2_to_H1': info['transferred_h2_to_h1'],
                'Trans_Cost_H1_to_H2': trans_cost_h1_to_h2,
                'Trans_Cost_H2_to_H1': trans_cost_h2_to_h1,
                'Network_Total_Cost': info['total_cost'],
            })

            state = next_state
            if done:
                break

        h1_records_all.append(pd.DataFrame(h1_daily))
        h2_records_all.append(pd.DataFrame(h2_daily))
        sc_records_all.append(pd.DataFrame(sc_daily))

    h1_seed_summaries = [compute_episode_summary(df, H1_COLS) for df in h1_records_all]
    h2_seed_summaries = [compute_episode_summary(df, H2_COLS) for df in h2_records_all]
    sc_seed_summaries = [compute_episode_summary(df, SC_COLS) for df in sc_records_all]

    h1_model_summary = aggregate_seed_summaries(h1_seed_summaries)
    h2_model_summary = aggregate_seed_summaries(h2_seed_summaries)
    sc_model_summary = aggregate_seed_summaries(sc_seed_summaries)

    raw_rows = []
    for i, seed in enumerate(eval_seeds):
        row = {'Train_Seed': train_seed, 'Eval_Seed': seed}
        for k, v in h1_seed_summaries[i].items():
            row[f'H1_{k}'] = v
        for k, v in h2_seed_summaries[i].items():
            row[f'H2_{k}'] = v
        for k, v in sc_seed_summaries[i].items():
            row[f'SC_{k}'] = v
        raw_rows.append(row)

    return h1_model_summary, h2_model_summary, sc_model_summary, raw_rows


# =============================================================================
# EXCEL BUILDER
# =============================================================================
def create_excel(all_h1_summaries, all_h2_summaries, all_sc_summaries, raw_rows, train_seeds):
    excel_path = f"{OUTPUT_PREFIX}_ALL_RESULTS.xlsx"
    print(f"\n{'='*80}")
    print(f"📊 CREATING EXCEL: {excel_path}")
    print(f"{'='*80}")

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

        def write_summary_sheet(dfs, sheet_name):
            combined = pd.concat(dfs, ignore_index=True)
            mean_cols = [c for c in combined.columns if c.endswith('_Mean')]
            result = pd.DataFrame({
                'Metric': [c.replace('_Mean', '') for c in mean_cols],
                'Mean': [combined[c].mean() for c in mean_cols],
                'Std': [combined[c].std() for c in mean_cols],
                'Min': [combined[c].min() for c in mean_cols],
                'Max': [combined[c].max() for c in mean_cols]
            })
            result.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"💾 Sheet '{sheet_name}': {len(result)} metrics")
            return result

        h1_summary_df = write_summary_sheet(all_h1_summaries, 'H1_Summary')
        h2_summary_df = write_summary_sheet(all_h2_summaries, 'H2_Summary')
        sc_summary_df = write_summary_sheet(all_sc_summaries, 'SC_Summary')

        # Master
        master_rows = []
        for _, row in h1_summary_df.iterrows():
            master_rows.append({'Table': 'H1', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        for _, row in h2_summary_df.iterrows():
            master_rows.append({'Table': 'H2', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        for _, row in sc_summary_df.iterrows():
            master_rows.append({'Table': 'SC', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        master_df = pd.DataFrame(master_rows)
        master_df.to_excel(writer, sheet_name='Master', index=False)
        print(f"💾 Sheet 'Master': {len(master_df)} rows")

        # Model_Comparison
        comparison_rows = []
        for seed, h1, h2, sc in zip(train_seeds, all_h1_summaries, all_h2_summaries, all_sc_summaries):
            comparison_rows.append({
                'Train_Seed': seed,
                'H1_Total_Cost': h1['Total_Cost_Mean'].values[0] if 'Total_Cost_Mean' in h1.columns else np.nan,
                'H2_Total_Cost': h2['Total_Cost_Mean'].values[0] if 'Total_Cost_Mean' in h2.columns else np.nan,
                'Network_Total_Cost': sc['Network_Total_Cost_Mean'].values[0] if 'Network_Total_Cost_Mean' in sc.columns else np.nan,
                'H1_Wastage_Rate': h1['Wastage_Rate_Pct_Mean'].values[0] if 'Wastage_Rate_Pct_Mean' in h1.columns else np.nan,
                'H2_Wastage_Rate': h2['Wastage_Rate_Pct_Mean'].values[0] if 'Wastage_Rate_Pct_Mean' in h2.columns else np.nan,
                'H1_Shortage_Rate': h1['Shortage_Rate_Pct_Mean'].values[0] if 'Shortage_Rate_Pct_Mean' in h1.columns else np.nan,
                'H2_Shortage_Rate': h2['Shortage_Rate_Pct_Mean'].values[0] if 'Shortage_Rate_Pct_Mean' in h2.columns else np.nan,
            })
        comparison_df = pd.DataFrame(comparison_rows)
        comparison_df.to_excel(writer, sheet_name='Model_Comparison', index=False)
        print(f"💾 Sheet 'Model_Comparison': {len(comparison_df)} models")

        # Raw_Episode_Data
        raw_df = pd.DataFrame(raw_rows)
        raw_df.to_excel(writer, sheet_name='Raw_Episode_Data', index=False)
        print(f"💾 Sheet 'Raw_Episode_Data': {len(raw_df)} rows")

    print(f"\n✅ Excel saved: {excel_path}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    TRAIN_SEEDS = [42, 123, 456, 789, 2025]

    all_rewards = {}
    trained_models = []

    # 1) Train 5 models
    for seed in TRAIN_SEEDS:
        path, rewards = train(train_seed=seed)
        trained_models.append((seed, path))
        all_rewards[seed] = rewards

    # 2) Combined learning curve (only this PNG)
    plot_combined_learning_curves(all_rewards)

    # 3) Evaluate each model on 50 seeds
    all_h1_summaries = []
    all_h2_summaries = []
    all_sc_summaries = []
    all_raw_rows = []

    for seed, path in trained_models:
        h1_mod, h2_mod, sc_mod, raw = evaluate_multi_seed(path, train_seed=seed, eval_seeds=range(1, 51))
        all_h1_summaries.append(h1_mod)
        all_h2_summaries.append(h2_mod)
        all_sc_summaries.append(sc_mod)
        all_raw_rows.extend(raw)

    # 4) One Excel file with 6 sheets
    create_excel(all_h1_summaries, all_h2_summaries, all_sc_summaries, all_raw_rows, TRAIN_SEEDS)


🚀 TRAINING SEED 42


Train Seed 42: 100%|██████████| 2000/2000 [04:25<00:00,  7.52it/s]



🚀 TRAINING SEED 123


Train Seed 123: 100%|██████████| 2000/2000 [04:24<00:00,  7.55it/s]



🚀 TRAINING SEED 456


...


🚀 EVALUATING MODEL (Train Seed 789) ON 50 SEEDS


Eval Train-Seed 789: 100%|██████████| 50/50 [00:03<00:00, 14.66it/s]



🚀 EVALUATING MODEL (Train Seed 2025) ON 50 SEEDS


Eval Train-Seed 2025: 100%|██████████| 50/50 [00:04<00:00, 11.97it/s]



📊 CREATING EXCEL: mappo_ALL_RESULTS.xlsx
💾 Sheet 'H1_Summary': 12 metrics
💾 Sheet 'H2_Summary': 12 metrics
💾 Sheet 'SC_Summary': 7 metrics
💾 Sheet 'Master': 31 rows
💾 Sheet 'Model_Comparison': 5 models
💾 Sheet 'Raw_Episode_Data': 250 rows

✅ Excel saved: mappo_ALL_RESULTS.xlsx


# Single Agent with Transshipment

In [2]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta
from torch.optim import Adam
from collections import deque
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION & HYPERPARAMETERS
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ZINB_H1_PARAMS = [{"phi": 0.25, "p": 0.48, "n": 15}]
ZINB_H2_PARAMS = [{"phi": 0.25, "p": 0.48, "n": 15}]

def _zinb_mean(params: dict) -> float:
    phi, p, n = params["phi"], params["p"], params["n"]
    return (1.0 - phi) * n * (1.0 - p) / p

def sample_zinb(phi: float, p: float, n: int) -> int:
    if np.random.random() < phi:
        return 0
    return int(np.random.negative_binomial(n, p))

def sample_hospital_demand(zinb_params: list, mult: float = 1.0) -> int:
    total = sum(sample_zinb(p["phi"], p["p"], p["n"]) for p in zinb_params)
    return max(0, int(total * mult))

ENV_CONFIG = {
    "max_capacity": 100,
    "shelf_life": 5,
    "order_cost": 1.0,
    "transshipment_cost": 1.0,
    "transshipment_fixed_cost": 5.0,
    "shortage_cost": 25.0,
    "wastage_cost": 15.0,
    "inventory_cost": 0.5,
    "episode_length": 60,
    "order_cap": 70,
    "trans_cap": 50,
    "transshipment_deadband": 0.1,
    "disruption_threshold": 1.2,
    "disruption_prob": 0.02,
    "disruption_min_len": 3,
    "disruption_max_len": 8,
    "disruption_mult": 1.4,
}

ORDER_SCHEDULE_H1 = {1, 4, 6}
ORDER_SCHEDULE_H2 = {0, 3, 5}

STATE_DIM = 26
ACTION_DIM = 3
STATE_SCALE = 70.0
REWARD_SCALE = 100.0

LR = 3e-4
GAMMA = 0.99
EPSILON = 0.2
EPOCHS = 4
GAE_LAMBDA = 0.95
UPDATE_EPISODES = 4
ENTROPY_COEF = 0.05
ENTROPY_DECAY = 0.999
MIN_ENTROPY = 0.005
TOTAL_EPISODES = 2000

OUTPUT_PREFIX = "single_agent_with_transshipment"

# =============================================================================
# ENVIRONMENT
# =============================================================================
class PlateletSupplyChainEnv:
    def __init__(self):
        for k, v in ENV_CONFIG.items():
            setattr(self, k, v)
        self.num_hospitals = 2
        self.demand_mean_h1 = sum(_zinb_mean(p) for p in ZINB_H1_PARAMS)
        self.demand_mean_h2 = sum(_zinb_mean(p) for p in ZINB_H2_PARAMS)
        self.reset()

    def _is_order_day_h1(self):
        return (self.day % 7) in ORDER_SCHEDULE_H1

    def _is_order_day_h2(self):
        return (self.day % 7) in ORDER_SCHEDULE_H2

    def _compute_disruption_flag(self, history):
        if len(history) < 5:
            return 0.0
        ma2 = np.mean(list(history)[-2:])
        ma30 = np.mean(list(history)[-30:])
        return 1.0 if (ma30 > 0 and ma2 > self.disruption_threshold * ma30) else 0.0

    def _get_observation(self):
        obs = np.zeros((STATE_DIM,), dtype=np.float32)
        idx = 0
        for h in range(self.num_hospitals):
            start = h * self.shelf_life
            for age in range(self.shelf_life):
                obs[idx] = self.state[start + age]
                idx += 1

        inv_h1 = np.sum(self.state[0:self.shelf_life])
        inv_h2 = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        obs[10] = self.day / self.episode_length
        obs[11] = self._compute_disruption_flag(self.demand_history_h1)
        obs[12] = self._compute_disruption_flag(self.demand_history_h2)
        obs[13] = inv_h1 / self.max_capacity
        obs[14] = inv_h2 / self.max_capacity
        obs[15] = 1.0 if self._is_order_day_h1() else 0.0
        obs[16] = 1.0 if self._is_order_day_h2() else 0.0
        obs[17] = (inv_h1 - inv_h2) / self.max_capacity
        obs[18] = self.crisis_counter_h1 / self.disruption_max_len
        obs[19] = self.crisis_counter_h2 / self.disruption_max_len

        d1 = np.mean(self.demand_history_h1[-7:]) if self.demand_history_h1 else self.demand_mean_h1
        d2 = np.mean(self.demand_history_h2[-7:]) if self.demand_history_h2 else self.demand_mean_h2
        obs[20] = min(inv_h1 / (d1 * 7 + 1e-8), 3.0)
        obs[21] = min(inv_h2 / (d2 * 7 + 1e-8), 3.0)

        if self.demand_history_h1:
            td1 = sum(self.demand_history_h1[-30:])
            obs[22] = sum(self.shortage_history_h1) / (td1 + 1e-8)
            obs[24] = sum(self.wastage_history_h1) / (td1 + 1e-8)
        if self.demand_history_h2:
            td2 = sum(self.demand_history_h2[-30:])
            obs[23] = sum(self.shortage_history_h2) / (td2 + 1e-8)
            obs[25] = sum(self.wastage_history_h2) / (td2 + 1e-8)
        return obs.copy()

    def reset(self):
        self.state = np.zeros((self.num_hospitals * self.shelf_life,), dtype=np.float32)
        self.day = 0
        self.demand_history_h1, self.demand_history_h2 = [], []
        self.shortage_history_h1, self.shortage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.wastage_history_h1, self.wastage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.disruption_counter_h1 = self.disruption_counter_h2 = 0
        self.crisis_counter_h1 = self.crisis_counter_h2 = 0
        return self._get_observation()

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)

        is_ord_h1 = self._is_order_day_h1()
        is_ord_h2 = self._is_order_day_h2()

        order_h1 = action[0] * self.order_cap if is_ord_h1 else 0.0
        order_h2 = action[1] * self.order_cap if is_ord_h2 else 0.0

        a_t = action[2] * 2.0 - 1.0
        tau = self.transshipment_deadband

        trans_h1_to_h2 = trans_h2_to_h1 = 0.0
        if a_t > tau:
            trans_h1_to_h2 = ((a_t - tau) / (1.0 - tau)) * self.trans_cap
        elif a_t < -tau:
            trans_h2_to_h1 = ((abs(a_t) - tau) / (1.0 - tau)) * self.trans_cap

        inv_h1_before = np.sum(self.state[0:self.shelf_life])
        inv_h2_before = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        actual_order_h1 = self._process_order(0, min(order_h1, max(0.0, self.max_capacity - inv_h1_before)))
        actual_order_h2 = self._process_order(1, min(order_h2, max(0.0, self.max_capacity - inv_h2_before)))

        actual_trans_h1_to_h2 = self._transfer_blood(0, 1, trans_h1_to_h2)
        actual_trans_h2_to_h1 = self._transfer_blood(1, 0, trans_h2_to_h1)

        if self.day >= 5:
            if self.disruption_counter_h1 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h1 = random.randint(self.disruption_min_len, self.disruption_max_len)
            if self.disruption_counter_h2 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h2 = random.randint(self.disruption_min_len, self.disruption_max_len)

        mult_h1 = self.disruption_mult if self.disruption_counter_h1 > 0 else 1.0
        mult_h2 = self.disruption_mult if self.disruption_counter_h2 > 0 else 1.0

        if self.disruption_counter_h1 > 0:
            self.disruption_counter_h1 -= 1
        if self.disruption_counter_h2 > 0:
            self.disruption_counter_h2 -= 1

        demand_h1 = sample_hospital_demand(ZINB_H1_PARAMS, mult=mult_h1)
        demand_h2 = sample_hospital_demand(ZINB_H2_PARAMS, mult=mult_h2)

        self.demand_history_h1.append(demand_h1)
        self.demand_history_h2.append(demand_h2)

        self.crisis_counter_h1 = min(self.crisis_counter_h1 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h1) > 0.5 else 0
        self.crisis_counter_h2 = min(self.crisis_counter_h2 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h2) > 0.5 else 0

        wastage_h1, shortage_h1, _ = self._fulfill_demand(0, demand_h1)
        wastage_h2, shortage_h2, _ = self._fulfill_demand(1, demand_h2)

        self.shortage_history_h1.append(shortage_h1)
        self.shortage_history_h2.append(shortage_h2)
        self.wastage_history_h1.append(wastage_h1)
        self.wastage_history_h2.append(wastage_h2)

        self._age_blood()

        inv_h1_after = np.sum(self.state[0:self.shelf_life])
        inv_h2_after = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        inventory_holding_cost = (inv_h1_before + inv_h2_before) * self.inventory_cost
        self.day += 1

        total_transferred = actual_trans_h1_to_h2 + actual_trans_h2_to_h1
        transshipment_cost = total_transferred * self.transshipment_cost + (self.transshipment_fixed_cost if total_transferred >= 1.0 else 0.0)
        wastage_cost = (wastage_h1 + wastage_h2) * self.wastage_cost
        shortage_cost = (shortage_h1 + shortage_h2) * self.shortage_cost
        order_cost = (actual_order_h1 + actual_order_h2) * self.order_cost

        total_cost = wastage_cost + shortage_cost + transshipment_cost + order_cost + inventory_holding_cost
        shared_reward = -total_cost / REWARD_SCALE

        info = {
            'order_mask': [1.0 if is_ord_h1 else 0.0, 1.0 if is_ord_h2 else 0.0],
            'demand_h1': demand_h1, 'demand_h2': demand_h2,
            'inventory_h1': inv_h1_after, 'inventory_h2': inv_h2_after,
            'order_h1': order_h1, 'order_h2': order_h2,
            'actual_order_h1': actual_order_h1, 'actual_order_h2': actual_order_h2,
            'transferred_h1_to_h2': actual_trans_h1_to_h2, 'transferred_h2_to_h1': actual_trans_h2_to_h1,
            'shortage_h1': shortage_h1, 'shortage_h2': shortage_h2,
            'wastage_h1': wastage_h1, 'wastage_h2': wastage_h2,
            'total_cost': total_cost, 'wastage_cost': wastage_cost,
            'shortage_cost': shortage_cost, 'transshipment_cost': transshipment_cost,
            'order_cost': order_cost, 'inventory_holding_cost': inventory_holding_cost,
            'disruption_active_h1': mult_h1 > 1.0, 'disruption_active_h2': mult_h2 > 1.0,
            'crisis_counter_h1': self.crisis_counter_h1, 'crisis_counter_h2': self.crisis_counter_h2,
            'inv_h1_before': inv_h1_before, 'inv_h2_before': inv_h2_before,
        }

        done = self.day >= self.episode_length
        return self._get_observation(), shared_reward, done, info

    def _process_order(self, hospital, amount):
        if amount <= 0:
            return 0.0
        start = hospital * self.shelf_life
        accepted = max(0.0, min(amount, self.max_capacity - np.sum(self.state[start:start + self.shelf_life])))
        self.state[start] += accepted
        return accepted

    def _transfer_blood(self, from_h, to_h, amount):
        if amount <= 0:
            return 0.0
        f_s, f_e = from_h * self.shelf_life, from_h * self.shelf_life + self.shelf_life
        t_s, t_e = to_h * self.shelf_life, to_h * self.shelf_life + self.shelf_life

        transferrable = max(0.0, min(amount, np.sum(self.state[f_s:f_e]), self.max_capacity - np.sum(self.state[t_s:t_e])))
        transferred = 0.0
        for i in range(f_e - 1, f_s - 1, -1):
            if self.state[i] > 0:
                to_transfer = min(self.state[i], transferrable - transferred)
                self.state[i] -= to_transfer
                self.state[t_s + (i - f_s)] += to_transfer
                transferred += to_transfer
                if transferred >= transferrable:
                    break
        return transferred

    def _fulfill_demand(self, hospital, demand):
        s_idx, e_idx = hospital * self.shelf_life, hospital * self.shelf_life + self.shelf_life
        unfulfilled = demand
        for i in range(e_idx - 1, s_idx - 1, -1):
            if self.state[i] > 0 and unfulfilled > 0:
                used = min(self.state[i], unfulfilled)
                self.state[i] -= used
                unfulfilled -= used
            if unfulfilled <= 1e-5:
                break
        wastage = self.state[e_idx - 1]
        self.state[e_idx - 1] = 0
        return wastage, unfulfilled, demand - unfulfilled

    def _age_blood(self):
        for h in range(self.num_hospitals):
            s_idx, e_idx = h * self.shelf_life, h * self.shelf_life + self.shelf_life
            self.state[s_idx + 1:e_idx] = self.state[s_idx:e_idx - 1]
            self.state[s_idx] = 0


# =============================================================================
# RL ALGORITHM (Single-Agent PPO)
# =============================================================================
class RunningMeanStd:
    def __init__(self, epsilon=1e-4, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = epsilon

    def update(self, x):
        batch_mean = np.mean(x)
        batch_var = np.var(x)
        batch_count = x.shape[0]
        self.update_from_moments(batch_mean, batch_var, batch_count)

    def update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        tot_count = self.count + batch_count
        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot_count
        new_var = M2 / tot_count
        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def normalize(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


class Actor(nn.Module):
    def __init__(self, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(STATE_DIM, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())
        self.alpha_head = nn.Linear(256, action_dim)
        self.beta_head = nn.Linear(256, action_dim)

    def forward(self, state):
        feat = self.net(state)
        alpha = torch.clamp(F.softplus(self.alpha_head(feat)) + 1.0, 1.0, 100.0)
        beta = torch.clamp(F.softplus(self.beta_head(feat)) + 1.0, 1.0, 100.0)
        return Beta(alpha, beta)

    def get_action(self, state):
        with torch.no_grad():
            dist = self.forward(state)
            action = dist.sample()
            safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
            return safe_act.squeeze(0).cpu().numpy(), dist.log_prob(safe_act).squeeze(0).cpu().numpy()

    def evaluate(self, state, action):
        dist = self.forward(state)
        safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
        return dist.log_prob(safe_act), dist.entropy()


class SingleAgentPPO:
    def __init__(self):
        self.actor = Actor(ACTION_DIM).to(DEVICE)
        self.critic = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        ).to(DEVICE)

        self.opt_a = Adam(self.actor.parameters(), lr=LR)
        self.opt_c = Adam(self.critic.parameters(), lr=LR)
        self.entropy_coef = ENTROPY_COEF
        self.rms = RunningMeanStd()

    def _norm(self, st):
        st = st.clone()
        st[:, :10] /= STATE_SCALE
        return st

    def get_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        act, logp = self.actor.get_action(st)
        return act, logp

    def get_deterministic_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        with torch.no_grad():
            act = self.actor.forward(st).mean.squeeze(0).cpu().numpy()
        return act

    def save(self, path):
        torch.save({
            'actor': self.actor.state_dict(),
            'critic': self.critic.state_dict()
        }, path)

    def load(self, path):
        ck = torch.load(path, map_location=DEVICE, weights_only=True)
        self.actor.load_state_dict(ck['actor'])
        self.critic.load_state_dict(ck['critic'])

    def update(self, s, a, lp, r, ns, d, masks):
        s = self._norm(torch.FloatTensor(np.array(s)).to(DEVICE))
        ns = self._norm(torch.FloatTensor(np.array(ns)).to(DEVICE))
        a = torch.FloatTensor(np.array(a)).to(DEVICE)
        lp = torch.FloatTensor(np.array(lp)).to(DEVICE)

        r_np = np.array(r)
        self.rms.update(r_np)
        r_norm = torch.FloatTensor(self.rms.normalize(r_np)).to(DEVICE)

        d = torch.FloatTensor(np.array(d)).to(DEVICE)
        masks = torch.FloatTensor(np.array(masks)).to(DEVICE)

        with torch.no_grad():
            vals = self.critic(s).squeeze()
            n_vals = self.critic(ns).squeeze()

        adv = torch.zeros_like(r_norm).to(DEVICE)
        gae = 0
        for t in reversed(range(len(r_norm))):
            delta = r_norm[t] + GAMMA * n_vals[t] * (1 - d[t]) - vals[t]
            gae = delta + GAMMA * GAE_LAMBDA * (1 - d[t]) * gae
            adv[t] = gae

        returns = adv + vals
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(EPOCHS):
            new_logp, ent = self.actor.evaluate(s, a)

            ratio = torch.exp(new_logp - lp.detach())

            # Apply mask only on first 2 dimensions (procurement)
            # Dimension 2 (logistics) is always active
            full_mask = torch.ones_like(ratio)
            full_mask[:, :2] = masks

            surr1 = ratio * adv.unsqueeze(1)
            surr2 = torch.clamp(ratio, 1 - EPSILON, 1 + EPSILON) * adv.unsqueeze(1)

            loss_a = -torch.min(surr1, surr2) * full_mask
            loss_a = loss_a.sum() / (full_mask.sum() + 1e-8)
            loss_a -= self.entropy_coef * (ent * full_mask).sum() / (full_mask.sum() + 1e-8)

            self.opt_a.zero_grad()
            loss_a.backward()
            nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
            self.opt_a.step()

            v_pred = self.critic(s).squeeze()
            loss_c = F.mse_loss(v_pred, returns)
            self.opt_c.zero_grad()
            loss_c.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
            self.opt_c.step()

        self.entropy_coef = max(MIN_ENTROPY, self.entropy_coef * ENTROPY_DECAY)


# =============================================================================
# TRAINING FUNCTION
# =============================================================================
def train(train_seed=42):
    print(f"\n{'='*80}")
    print(f"🚀 TRAINING SEED {train_seed} — Single-Agent PPO (With Transshipment)")
    print(f"{'='*80}")

    random.seed(train_seed)
    np.random.seed(train_seed)
    torch.manual_seed(train_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(train_seed)

    env = PlateletSupplyChainEnv()
    agent = SingleAgentPPO()

    buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []
    model_path = None

    for ep in tqdm(range(TOTAL_EPISODES), desc=f"Train Seed {train_seed}"):
        s = env.reset()
        ep_r = 0
        for step in range(env.episode_length):
            act, lp = agent.get_action(s)
            ns, r, d, info = env.step(act)

            buf_s.append(s)
            buf_a.append(act)
            buf_lp.append(lp)
            buf_r.append(r)
            buf_ns.append(ns)
            buf_d.append(d)
            buf_m.append(info['order_mask'])

            s = ns
            ep_r += r
            if d:
                break

        rewards_log.append(ep_r)

        if (ep + 1) % UPDATE_EPISODES == 0:
            agent.update(buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []

        if ep >= 100:
            avg_r = np.mean(rewards_log[-100:])
            if avg_r > best_reward:
                best_reward = avg_r
                model_path = f"{OUTPUT_PREFIX}_seed{train_seed}.pt"
                agent.save(model_path)

    return model_path, rewards_log


# =============================================================================
# COMBINED LEARNING CURVE
# =============================================================================
def plot_combined_learning_curves(all_rewards):
    print(f"\n{'='*80}")
    print(f"📈 GENERATING COMBINED LEARNING CURVE")
    print(f"{'='*80}")

    plt.figure(figsize=(12, 6))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    for idx, (seed, rewards) in enumerate(all_rewards.items()):
        if len(rewards) >= 50:
            smoothed = np.convolve(rewards, np.ones(50)/50, mode='valid')
            episodes = np.arange(49, len(rewards))
            plt.plot(episodes, smoothed, color=colors[idx % len(colors)],
                     linewidth=2.5, label=f'Model {idx+1} (Seed {seed})')
        plt.plot(rewards, alpha=0.08, color=colors[idx % len(colors)])

    plt.xlabel("Episode", fontsize=13, fontweight='bold')
    plt.ylabel("Episode Reward", fontsize=13, fontweight='bold')
    plt.title("Single-Agent PPO Training Curves — Platelet Supply Chain (With Transshipment)", fontsize=15, fontweight='bold')
    plt.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_PREFIX}_combined_learning_curves.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"💾 Saved: {OUTPUT_PREFIX}_combined_learning_curves.png")


# =============================================================================
# EVALUATION HELPERS
# =============================================================================
H1_COLS = ['Total_Demand', 'Fulfilled_Demand', 'Unfulfilled_Demand', 'Wastage_Units',
           'Shortage_Units', 'Wastage_Rate_Pct', 'Shortage_Rate_Pct', 'Wastage_Cost',
           'Shortage_Cost', 'Holding_Cost', 'Order_Cost', 'Total_Cost']
H2_COLS = H1_COLS
SC_COLS = ['Total_Wastage', 'Total_Shortage', 'Trans_H1_to_H2', 'Trans_H2_to_H1',
           'Trans_Cost_H1_to_H2', 'Trans_Cost_H2_to_H1', 'Network_Total_Cost']


def compute_episode_summary(df, cols):
    s = {}
    for c in cols:
        if 'Cost' in c or 'Units' in c or 'Demand' in c or 'Trans' in c:
            s[c] = df[c].sum()
        else:
            s[c] = df[c].mean()
    return s


def aggregate_seed_summaries(seed_summaries):
    df = pd.DataFrame(seed_summaries)
    result = {}
    for c in df.columns:
        result[f'{c}_Mean'] = df[c].mean()
        result[f'{c}_Std'] = df[c].std()
    return pd.DataFrame([result])


def evaluate_multi_seed(model_path, train_seed, eval_seeds=range(1, 51)):
    print(f"\n{'='*80}")
    print(f"🚀 EVALUATING MODEL (Train Seed {train_seed}) ON {len(eval_seeds)} SEEDS")
    print(f"{'='*80}")

    env = PlateletSupplyChainEnv()
    agent = SingleAgentPPO()
    agent.load(model_path)
    agent.actor.eval()

    h1_records_all = []
    h2_records_all = []
    sc_records_all = []

    for seed in tqdm(eval_seeds, desc=f"Eval Train-Seed {train_seed}"):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        state = env.reset()
        h1_daily = []
        h2_daily = []
        sc_daily = []

        for day in range(env.episode_length):
            action = agent.get_deterministic_action(state)
            next_state, reward, done, info = env.step(action)

            d1 = info['demand_h1']
            s1 = info['shortage_h1']
            w1 = info['wastage_h1']
            fulfilled1 = d1 - s1
            inv_h1_before = info['inv_h1_before']
            inv_h2_before = info['inv_h2_before']
            total_inv_before = inv_h1_before + inv_h2_before

            h1_hold = (inv_h1_before / total_inv_before) * info['inventory_holding_cost'] if total_inv_before > 0 else 0.0

            h1_daily.append({
                'Day': day + 1,
                'Total_Demand': d1,
                'Fulfilled_Demand': fulfilled1,
                'Unfulfilled_Demand': s1,
                'Wastage_Units': w1,
                'Shortage_Units': s1,
                'Wastage_Rate_Pct': (w1 / d1 * 100) if d1 > 0 else 0.0,
                'Shortage_Rate_Pct': (s1 / d1 * 100) if d1 > 0 else 0.0,
                'Wastage_Cost': w1 * env.wastage_cost,
                'Shortage_Cost': s1 * env.shortage_cost,
                'Holding_Cost': h1_hold,
                'Order_Cost': info['actual_order_h1'] * env.order_cost,
                'Total_Cost': (w1 * env.wastage_cost) + (s1 * env.shortage_cost) + h1_hold + (info['actual_order_h1'] * env.order_cost),
            })

            d2 = info['demand_h2']
            s2 = info['shortage_h2']
            w2 = info['wastage_h2']
            fulfilled2 = d2 - s2

            h2_hold = (inv_h2_before / total_inv_before) * info['inventory_holding_cost'] if total_inv_before > 0 else 0.0

            h2_daily.append({
                'Day': day + 1,
                'Total_Demand': d2,
                'Fulfilled_Demand': fulfilled2,
                'Unfulfilled_Demand': s2,
                'Wastage_Units': w2,
                'Shortage_Units': s2,
                'Wastage_Rate_Pct': (w2 / d2 * 100) if d2 > 0 else 0.0,
                'Shortage_Rate_Pct': (s2 / d2 * 100) if d2 > 0 else 0.0,
                'Wastage_Cost': w2 * env.wastage_cost,
                'Shortage_Cost': s2 * env.shortage_cost,
                'Holding_Cost': h2_hold,
                'Order_Cost': info['actual_order_h2'] * env.order_cost,
                'Total_Cost': (w2 * env.wastage_cost) + (s2 * env.shortage_cost) + h2_hold + (info['actual_order_h2'] * env.order_cost),
            })

            total_trans = info['transferred_h1_to_h2'] + info['transferred_h2_to_h1']
            trans_cost_total = info['transshipment_cost']

            if total_trans > 0:
                trans_cost_h1_to_h2 = (info['transferred_h1_to_h2'] / total_trans) * trans_cost_total
                trans_cost_h2_to_h1 = (info['transferred_h2_to_h1'] / total_trans) * trans_cost_total
            else:
                trans_cost_h1_to_h2 = 0.0
                trans_cost_h2_to_h1 = 0.0

            sc_daily.append({
                'Day': day + 1,
                'Total_Wastage': w1 + w2,
                'Total_Shortage': s1 + s2,
                'Trans_H1_to_H2': info['transferred_h1_to_h2'],
                'Trans_H2_to_H1': info['transferred_h2_to_h1'],
                'Trans_Cost_H1_to_H2': trans_cost_h1_to_h2,
                'Trans_Cost_H2_to_H1': trans_cost_h2_to_h1,
                'Network_Total_Cost': info['total_cost'],
            })

            state = next_state
            if done:
                break

        h1_records_all.append(pd.DataFrame(h1_daily))
        h2_records_all.append(pd.DataFrame(h2_daily))
        sc_records_all.append(pd.DataFrame(sc_daily))

    h1_seed_summaries = [compute_episode_summary(df, H1_COLS) for df in h1_records_all]
    h2_seed_summaries = [compute_episode_summary(df, H2_COLS) for df in h2_records_all]
    sc_seed_summaries = [compute_episode_summary(df, SC_COLS) for df in sc_records_all]

    h1_model_summary = aggregate_seed_summaries(h1_seed_summaries)
    h2_model_summary = aggregate_seed_summaries(h2_seed_summaries)
    sc_model_summary = aggregate_seed_summaries(sc_seed_summaries)

    raw_rows = []
    for i, seed in enumerate(eval_seeds):
        row = {'Train_Seed': train_seed, 'Eval_Seed': seed}
        for k, v in h1_seed_summaries[i].items():
            row[f'H1_{k}'] = v
        for k, v in h2_seed_summaries[i].items():
            row[f'H2_{k}'] = v
        for k, v in sc_seed_summaries[i].items():
            row[f'SC_{k}'] = v
        raw_rows.append(row)

    return h1_model_summary, h2_model_summary, sc_model_summary, raw_rows


# =============================================================================
# EXCEL BUILDER
# =============================================================================
def create_excel(all_h1_summaries, all_h2_summaries, all_sc_summaries, raw_rows, train_seeds):
    excel_path = f"{OUTPUT_PREFIX}_ALL_RESULTS.xlsx"
    print(f"\n{'='*80}")
    print(f"📊 CREATING EXCEL: {excel_path}")
    print(f"{'='*80}")

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

        def write_summary_sheet(dfs, sheet_name):
            combined = pd.concat(dfs, ignore_index=True)
            mean_cols = [c for c in combined.columns if c.endswith('_Mean')]
            result = pd.DataFrame({
                'Metric': [c.replace('_Mean', '') for c in mean_cols],
                'Mean': [combined[c].mean() for c in mean_cols],
                'Std': [combined[c].std() for c in mean_cols],
                'Min': [combined[c].min() for c in mean_cols],
                'Max': [combined[c].max() for c in mean_cols]
            })
            result.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"💾 Sheet '{sheet_name}': {len(result)} metrics")
            return result

        h1_summary_df = write_summary_sheet(all_h1_summaries, 'H1_Summary')
        h2_summary_df = write_summary_sheet(all_h2_summaries, 'H2_Summary')
        sc_summary_df = write_summary_sheet(all_sc_summaries, 'SC_Summary')

        # Master
        master_rows = []
        for _, row in h1_summary_df.iterrows():
            master_rows.append({'Table': 'H1', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        for _, row in h2_summary_df.iterrows():
            master_rows.append({'Table': 'H2', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        for _, row in sc_summary_df.iterrows():
            master_rows.append({'Table': 'SC', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        master_df = pd.DataFrame(master_rows)
        master_df.to_excel(writer, sheet_name='Master', index=False)
        print(f"💾 Sheet 'Master': {len(master_df)} rows")

        # Model_Comparison
        comparison_rows = []
        for seed, h1, h2, sc in zip(train_seeds, all_h1_summaries, all_h2_summaries, all_sc_summaries):
            comparison_rows.append({
                'Train_Seed': seed,
                'H1_Total_Cost': h1['Total_Cost_Mean'].values[0] if 'Total_Cost_Mean' in h1.columns else np.nan,
                'H2_Total_Cost': h2['Total_Cost_Mean'].values[0] if 'Total_Cost_Mean' in h2.columns else np.nan,
                'Network_Total_Cost': sc['Network_Total_Cost_Mean'].values[0] if 'Network_Total_Cost_Mean' in sc.columns else np.nan,
                'H1_Wastage_Rate': h1['Wastage_Rate_Pct_Mean'].values[0] if 'Wastage_Rate_Pct_Mean' in h1.columns else np.nan,
                'H2_Wastage_Rate': h2['Wastage_Rate_Pct_Mean'].values[0] if 'Wastage_Rate_Pct_Mean' in h2.columns else np.nan,
                'H1_Shortage_Rate': h1['Shortage_Rate_Pct_Mean'].values[0] if 'Shortage_Rate_Pct_Mean' in h1.columns else np.nan,
                'H2_Shortage_Rate': h2['Shortage_Rate_Pct_Mean'].values[0] if 'Shortage_Rate_Pct_Mean' in h2.columns else np.nan,
            })
        comparison_df = pd.DataFrame(comparison_rows)
        comparison_df.to_excel(writer, sheet_name='Model_Comparison', index=False)
        print(f"💾 Sheet 'Model_Comparison': {len(comparison_df)} models")

        # Raw_Episode_Data
        raw_df = pd.DataFrame(raw_rows)
        raw_df.to_excel(writer, sheet_name='Raw_Episode_Data', index=False)
        print(f"💾 Sheet 'Raw_Episode_Data': {len(raw_df)} rows")

    print(f"\n✅ Excel saved: {excel_path}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    TRAIN_SEEDS = [42, 123, 456, 789, 2025]

    all_rewards = {}
    trained_models = []

    # 1) Train 5 models
    for seed in TRAIN_SEEDS:
        path, rewards = train(train_seed=seed)
        trained_models.append((seed, path))
        all_rewards[seed] = rewards

    # 2) Combined learning curve (only this PNG)
    plot_combined_learning_curves(all_rewards)

    # 3) Evaluate each model on 50 seeds
    all_h1_summaries = []
    all_h2_summaries = []
    all_sc_summaries = []
    all_raw_rows = []

    for seed, path in trained_models:
        h1_mod, h2_mod, sc_mod, raw = evaluate_multi_seed(path, train_seed=seed, eval_seeds=range(1, 51))
        all_h1_summaries.append(h1_mod)
        all_h2_summaries.append(h2_mod)
        all_sc_summaries.append(sc_mod)
        all_raw_rows.extend(raw)

    # 4) One Excel file with 6 sheets
    create_excel(all_h1_summaries, all_h2_summaries, all_sc_summaries, all_raw_rows, TRAIN_SEEDS)


🚀 TRAINING SEED 42 — Single-Agent PPO (With Transshipment)


Train Seed 42: 100%|██████████| 2000/2000 [03:04<00:00, 10.82it/s]



🚀 TRAINING SEED 123 — Single-Agent PPO (With Transshipment)


Train Seed 123: 100%|██████████| 2000/2000 [02:50<00:00, 11.74it/s]



🚀 TRAINING SEED 456 — Single-Agent PPO (With Transshipment)


...


🚀 EVALUATING MODEL (Train Seed 789) ON 50 SEEDS


Eval Train-Seed 789: 100%|██████████| 50/50 [00:02<00:00, 20.88it/s]



🚀 EVALUATING MODEL (Train Seed 2025) ON 50 SEEDS


Eval Train-Seed 2025: 100%|██████████| 50/50 [00:02<00:00, 21.00it/s]



📊 CREATING EXCEL: single_agent_with_transshipment_ALL_RESULTS.xlsx
💾 Sheet 'H1_Summary': 12 metrics
💾 Sheet 'H2_Summary': 12 metrics
💾 Sheet 'SC_Summary': 7 metrics
💾 Sheet 'Master': 31 rows
💾 Sheet 'Model_Comparison': 5 models
💾 Sheet 'Raw_Episode_Data': 250 rows

✅ Excel saved: single_agent_with_transshipment_ALL_RESULTS.xlsx


# Single Agent without Transshipment

In [3]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta
from torch.optim import Adam
from collections import deque
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION & HYPERPARAMETERS
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ZINB_H1_PARAMS = [{"phi": 0.25, "p": 0.48, "n": 15}]
ZINB_H2_PARAMS = [{"phi": 0.25, "p": 0.48, "n": 15}]

def _zinb_mean(params: dict) -> float:
    phi, p, n = params["phi"], params["p"], params["n"]
    return (1.0 - phi) * n * (1.0 - p) / p

def sample_zinb(phi: float, p: float, n: int) -> int:
    if np.random.random() < phi:
        return 0
    return int(np.random.negative_binomial(n, p))

def sample_hospital_demand(zinb_params: list, mult: float = 1.0) -> int:
    total = sum(sample_zinb(p["phi"], p["p"], p["n"]) for p in zinb_params)
    return max(0, int(total * mult))

ENV_CONFIG = {
    "max_capacity": 100,
    "shelf_life": 5,
    "order_cost": 1.0,
    "transshipment_cost": 1.0,
    "transshipment_fixed_cost": 5.0,
    "shortage_cost": 25.0,
    "wastage_cost": 15.0,
    "inventory_cost": 0.5,
    "episode_length": 60,
    "order_cap": 70,
    "trans_cap": 50,
    "transshipment_deadband": 0.1,
    "disruption_threshold": 1.2,
    "disruption_prob": 0.02,
    "disruption_min_len": 3,
    "disruption_max_len": 8,
    "disruption_mult": 1.4,
}

ORDER_SCHEDULE_H1 = {1, 4, 6}
ORDER_SCHEDULE_H2 = {0, 3, 5}

STATE_DIM = 26
ACTION_DIM = 2
STATE_SCALE = 70.0
REWARD_SCALE = 100.0

LR = 3e-4
GAMMA = 0.99
EPSILON = 0.2
EPOCHS = 4
GAE_LAMBDA = 0.95
UPDATE_EPISODES = 4
ENTROPY_COEF = 0.05
ENTROPY_DECAY = 0.999
MIN_ENTROPY = 0.005
TOTAL_EPISODES = 2000

OUTPUT_PREFIX = "single_agent_without_transshipment"

# =============================================================================
# ENVIRONMENT (NO TRANSSHIPMENT)
# =============================================================================
class PlateletSupplyChainEnv:
    def __init__(self):
        for k, v in ENV_CONFIG.items():
            setattr(self, k, v)
        self.num_hospitals = 2
        self.demand_mean_h1 = sum(_zinb_mean(p) for p in ZINB_H1_PARAMS)
        self.demand_mean_h2 = sum(_zinb_mean(p) for p in ZINB_H2_PARAMS)
        self.reset()

    def _is_order_day_h1(self):
        return (self.day % 7) in ORDER_SCHEDULE_H1

    def _is_order_day_h2(self):
        return (self.day % 7) in ORDER_SCHEDULE_H2

    def _compute_disruption_flag(self, history):
        if len(history) < 5:
            return 0.0
        ma2 = np.mean(list(history)[-2:])
        ma30 = np.mean(list(history)[-30:])
        return 1.0 if (ma30 > 0 and ma2 > self.disruption_threshold * ma30) else 0.0

    def _get_observation(self):
        obs = np.zeros((STATE_DIM,), dtype=np.float32)
        idx = 0
        for h in range(self.num_hospitals):
            start = h * self.shelf_life
            for age in range(self.shelf_life):
                obs[idx] = self.state[start + age]
                idx += 1

        inv_h1 = np.sum(self.state[0:self.shelf_life])
        inv_h2 = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        obs[10] = self.day / self.episode_length
        obs[11] = self._compute_disruption_flag(self.demand_history_h1)
        obs[12] = self._compute_disruption_flag(self.demand_history_h2)
        obs[13] = inv_h1 / self.max_capacity
        obs[14] = inv_h2 / self.max_capacity
        obs[15] = 1.0 if self._is_order_day_h1() else 0.0
        obs[16] = 1.0 if self._is_order_day_h2() else 0.0
        obs[17] = (inv_h1 - inv_h2) / self.max_capacity
        obs[18] = self.crisis_counter_h1 / self.disruption_max_len
        obs[19] = self.crisis_counter_h2 / self.disruption_max_len

        d1 = np.mean(self.demand_history_h1[-7:]) if self.demand_history_h1 else self.demand_mean_h1
        d2 = np.mean(self.demand_history_h2[-7:]) if self.demand_history_h2 else self.demand_mean_h2
        obs[20] = min(inv_h1 / (d1 * 7 + 1e-8), 3.0)
        obs[21] = min(inv_h2 / (d2 * 7 + 1e-8), 3.0)

        if self.demand_history_h1:
            td1 = sum(self.demand_history_h1[-30:])
            obs[22] = sum(self.shortage_history_h1) / (td1 + 1e-8)
            obs[24] = sum(self.wastage_history_h1) / (td1 + 1e-8)
        if self.demand_history_h2:
            td2 = sum(self.demand_history_h2[-30:])
            obs[23] = sum(self.shortage_history_h2) / (td2 + 1e-8)
            obs[25] = sum(self.wastage_history_h2) / (td2 + 1e-8)
        return obs.copy()

    def reset(self):
        self.state = np.zeros((self.num_hospitals * self.shelf_life,), dtype=np.float32)
        self.day = 0
        self.demand_history_h1, self.demand_history_h2 = [], []
        self.shortage_history_h1, self.shortage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.wastage_history_h1, self.wastage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.disruption_counter_h1 = self.disruption_counter_h2 = 0
        self.crisis_counter_h1 = self.crisis_counter_h2 = 0
        return self._get_observation()

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)

        is_ord_h1 = self._is_order_day_h1()
        is_ord_h2 = self._is_order_day_h2()

        order_h1 = action[0] * self.order_cap if is_ord_h1 else 0.0
        order_h2 = action[1] * self.order_cap if is_ord_h2 else 0.0

        inv_h1_before = np.sum(self.state[0:self.shelf_life])
        inv_h2_before = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        actual_order_h1 = self._process_order(0, min(order_h1, max(0.0, self.max_capacity - inv_h1_before)))
        actual_order_h2 = self._process_order(1, min(order_h2, max(0.0, self.max_capacity - inv_h2_before)))

        # No transshipment
        actual_trans_h1_to_h2 = 0.0
        actual_trans_h2_to_h1 = 0.0

        if self.day >= 5:
            if self.disruption_counter_h1 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h1 = random.randint(self.disruption_min_len, self.disruption_max_len)
            if self.disruption_counter_h2 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h2 = random.randint(self.disruption_min_len, self.disruption_max_len)

        mult_h1 = self.disruption_mult if self.disruption_counter_h1 > 0 else 1.0
        mult_h2 = self.disruption_mult if self.disruption_counter_h2 > 0 else 1.0

        if self.disruption_counter_h1 > 0:
            self.disruption_counter_h1 -= 1
        if self.disruption_counter_h2 > 0:
            self.disruption_counter_h2 -= 1

        demand_h1 = sample_hospital_demand(ZINB_H1_PARAMS, mult=mult_h1)
        demand_h2 = sample_hospital_demand(ZINB_H2_PARAMS, mult=mult_h2)

        self.demand_history_h1.append(demand_h1)
        self.demand_history_h2.append(demand_h2)

        self.crisis_counter_h1 = min(self.crisis_counter_h1 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h1) > 0.5 else 0
        self.crisis_counter_h2 = min(self.crisis_counter_h2 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h2) > 0.5 else 0

        wastage_h1, shortage_h1, _ = self._fulfill_demand(0, demand_h1)
        wastage_h2, shortage_h2, _ = self._fulfill_demand(1, demand_h2)

        self.shortage_history_h1.append(shortage_h1)
        self.shortage_history_h2.append(shortage_h2)
        self.wastage_history_h1.append(wastage_h1)
        self.wastage_history_h2.append(wastage_h2)

        self._age_blood()

        inv_h1_after = np.sum(self.state[0:self.shelf_life])
        inv_h2_after = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        inventory_holding_cost = (inv_h1_before + inv_h2_before) * self.inventory_cost
        self.day += 1

        transshipment_cost = 0.0
        wastage_cost = (wastage_h1 + wastage_h2) * self.wastage_cost
        shortage_cost = (shortage_h1 + shortage_h2) * self.shortage_cost
        order_cost = (actual_order_h1 + actual_order_h2) * self.order_cost

        total_cost = wastage_cost + shortage_cost + order_cost + inventory_holding_cost
        shared_reward = -total_cost / REWARD_SCALE

        info = {
            'order_mask': [1.0 if is_ord_h1 else 0.0, 1.0 if is_ord_h2 else 0.0],
            'demand_h1': demand_h1, 'demand_h2': demand_h2,
            'inventory_h1': inv_h1_after, 'inventory_h2': inv_h2_after,
            'order_h1': order_h1, 'order_h2': order_h2,
            'actual_order_h1': actual_order_h1, 'actual_order_h2': actual_order_h2,
            'transferred_h1_to_h2': actual_trans_h1_to_h2, 'transferred_h2_to_h1': actual_trans_h2_to_h1,
            'shortage_h1': shortage_h1, 'shortage_h2': shortage_h2,
            'wastage_h1': wastage_h1, 'wastage_h2': wastage_h2,
            'total_cost': total_cost, 'wastage_cost': wastage_cost,
            'shortage_cost': shortage_cost, 'transshipment_cost': transshipment_cost,
            'order_cost': order_cost, 'inventory_holding_cost': inventory_holding_cost,
            'disruption_active_h1': mult_h1 > 1.0, 'disruption_active_h2': mult_h2 > 1.0,
            'crisis_counter_h1': self.crisis_counter_h1, 'crisis_counter_h2': self.crisis_counter_h2,
            'inv_h1_before': inv_h1_before, 'inv_h2_before': inv_h2_before,
        }

        done = self.day >= self.episode_length
        return self._get_observation(), shared_reward, done, info

    def _process_order(self, hospital, amount):
        if amount <= 0:
            return 0.0
        start = hospital * self.shelf_life
        accepted = max(0.0, min(amount, self.max_capacity - np.sum(self.state[start:start + self.shelf_life])))
        self.state[start] += accepted
        return accepted

    def _fulfill_demand(self, hospital, demand):
        s_idx, e_idx = hospital * self.shelf_life, hospital * self.shelf_life + self.shelf_life
        unfulfilled = demand
        for i in range(e_idx - 1, s_idx - 1, -1):
            if self.state[i] > 0 and unfulfilled > 0:
                used = min(self.state[i], unfulfilled)
                self.state[i] -= used
                unfulfilled -= used
            if unfulfilled <= 1e-5:
                break
        wastage = self.state[e_idx - 1]
        self.state[e_idx - 1] = 0
        return wastage, unfulfilled, demand - unfulfilled

    def _age_blood(self):
        for h in range(self.num_hospitals):
            s_idx, e_idx = h * self.shelf_life, h * self.shelf_life + self.shelf_life
            self.state[s_idx + 1:e_idx] = self.state[s_idx:e_idx - 1]
            self.state[s_idx] = 0


# =============================================================================
# RL ALGORITHM (Single-Agent PPO)
# =============================================================================
class RunningMeanStd:
    def __init__(self, epsilon=1e-4, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = epsilon

    def update(self, x):
        batch_mean = np.mean(x)
        batch_var = np.var(x)
        batch_count = x.shape[0]
        self.update_from_moments(batch_mean, batch_var, batch_count)

    def update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        tot_count = self.count + batch_count
        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot_count
        new_var = M2 / tot_count
        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def normalize(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


class Actor(nn.Module):
    def __init__(self, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(STATE_DIM, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())
        self.alpha_head = nn.Linear(256, action_dim)
        self.beta_head = nn.Linear(256, action_dim)

    def forward(self, state):
        feat = self.net(state)
        alpha = torch.clamp(F.softplus(self.alpha_head(feat)) + 1.0, 1.0, 100.0)
        beta = torch.clamp(F.softplus(self.beta_head(feat)) + 1.0, 1.0, 100.0)
        return Beta(alpha, beta)

    def get_action(self, state):
        with torch.no_grad():
            dist = self.forward(state)
            action = dist.sample()
            safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
            return safe_act.squeeze(0).cpu().numpy(), dist.log_prob(safe_act).squeeze(0).cpu().numpy()

    def evaluate(self, state, action):
        dist = self.forward(state)
        safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
        return dist.log_prob(safe_act), dist.entropy()


class SingleAgentPPO:
    def __init__(self):
        self.actor = Actor(ACTION_DIM).to(DEVICE)
        self.critic = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        ).to(DEVICE)

        self.opt_a = Adam(self.actor.parameters(), lr=LR)
        self.opt_c = Adam(self.critic.parameters(), lr=LR)
        self.entropy_coef = ENTROPY_COEF
        self.rms = RunningMeanStd()

    def _norm(self, st):
        st = st.clone()
        st[:, :10] /= STATE_SCALE
        return st

    def get_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        act, logp = self.actor.get_action(st)
        return act, logp

    def get_deterministic_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        with torch.no_grad():
            act = self.actor.forward(st).mean.squeeze(0).cpu().numpy()
        return act

    def save(self, path):
        torch.save({
            'actor': self.actor.state_dict(),
            'critic': self.critic.state_dict()
        }, path)

    def load(self, path):
        ck = torch.load(path, map_location=DEVICE, weights_only=True)
        self.actor.load_state_dict(ck['actor'])
        self.critic.load_state_dict(ck['critic'])

    def update(self, s, a, lp, r, ns, d, masks):
        s = self._norm(torch.FloatTensor(np.array(s)).to(DEVICE))
        ns = self._norm(torch.FloatTensor(np.array(ns)).to(DEVICE))
        a = torch.FloatTensor(np.array(a)).to(DEVICE)
        lp = torch.FloatTensor(np.array(lp)).to(DEVICE)

        r_np = np.array(r)
        self.rms.update(r_np)
        r_norm = torch.FloatTensor(self.rms.normalize(r_np)).to(DEVICE)

        d = torch.FloatTensor(np.array(d)).to(DEVICE)
        masks = torch.FloatTensor(np.array(masks)).to(DEVICE)

        with torch.no_grad():
            vals = self.critic(s).squeeze()
            n_vals = self.critic(ns).squeeze()

        adv = torch.zeros_like(r_norm).to(DEVICE)
        gae = 0
        for t in reversed(range(len(r_norm))):
            delta = r_norm[t] + GAMMA * n_vals[t] * (1 - d[t]) - vals[t]
            gae = delta + GAMMA * GAE_LAMBDA * (1 - d[t]) * gae
            adv[t] = gae

        returns = adv + vals
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(EPOCHS):
            new_logp, ent = self.actor.evaluate(s, a)

            ratio = torch.exp(new_logp - lp.detach())
            surr1 = ratio * adv.unsqueeze(1)
            surr2 = torch.clamp(ratio, 1 - EPSILON, 1 + EPSILON) * adv.unsqueeze(1)

            loss_a = -torch.min(surr1, surr2) * masks
            loss_a = loss_a.sum() / (masks.sum() + 1e-8)
            loss_a -= self.entropy_coef * (ent * masks).sum() / (masks.sum() + 1e-8)

            self.opt_a.zero_grad()
            loss_a.backward()
            nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
            self.opt_a.step()

            v_pred = self.critic(s).squeeze()
            loss_c = F.mse_loss(v_pred, returns)
            self.opt_c.zero_grad()
            loss_c.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
            self.opt_c.step()

        self.entropy_coef = max(MIN_ENTROPY, self.entropy_coef * ENTROPY_DECAY)


# =============================================================================
# TRAINING FUNCTION
# =============================================================================
def train(train_seed=42):
    print(f"\n{'='*80}")
    print(f"🚀 TRAINING SEED {train_seed} — Single-Agent PPO (No Transshipment)")
    print(f"{'='*80}")

    random.seed(train_seed)
    np.random.seed(train_seed)
    torch.manual_seed(train_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(train_seed)

    env = PlateletSupplyChainEnv()
    ppo = SingleAgentPPO()

    buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []
    model_path = None

    for ep in tqdm(range(TOTAL_EPISODES), desc=f"Train Seed {train_seed}"):
        s = env.reset()
        ep_r = 0
        for step in range(env.episode_length):
            act, lp = ppo.get_action(s)
            ns, r, d, info = env.step(act)

            buf_s.append(s)
            buf_a.append(act)
            buf_lp.append(lp)
            buf_r.append(r)
            buf_ns.append(ns)
            buf_d.append(d)
            buf_m.append(info['order_mask'])

            s = ns
            ep_r += r
            if d:
                break

        rewards_log.append(ep_r)

        if (ep + 1) % UPDATE_EPISODES == 0:
            ppo.update(buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []

        if ep >= 100:
            avg_r = np.mean(rewards_log[-100:])
            if avg_r > best_reward:
                best_reward = avg_r
                model_path = f"{OUTPUT_PREFIX}_seed{train_seed}.pt"
                ppo.save(model_path)

    return model_path, rewards_log


# =============================================================================
# COMBINED LEARNING CURVE
# =============================================================================
def plot_combined_learning_curves(all_rewards):
    print(f"\n{'='*80}")
    print(f"📈 GENERATING COMBINED LEARNING CURVE")
    print(f"{'='*80}")

    plt.figure(figsize=(12, 6))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    for idx, (seed, rewards) in enumerate(all_rewards.items()):
        if len(rewards) >= 50:
            smoothed = np.convolve(rewards, np.ones(50)/50, mode='valid')
            episodes = np.arange(49, len(rewards))
            plt.plot(episodes, smoothed, color=colors[idx % len(colors)],
                     linewidth=2.5, label=f'Model {idx+1} (Seed {seed})')
        plt.plot(rewards, alpha=0.08, color=colors[idx % len(colors)])

    plt.xlabel("Episode", fontsize=13, fontweight='bold')
    plt.ylabel("Episode Reward", fontsize=13, fontweight='bold')
    plt.title("Single-Agent PPO Training Curves — Platelet Supply Chain (No Transshipment)", fontsize=15, fontweight='bold')
    plt.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_PREFIX}_combined_learning_curves.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"💾 Saved: {OUTPUT_PREFIX}_combined_learning_curves.png")


# =============================================================================
# EVALUATION HELPERS
# =============================================================================
H1_COLS = ['Total_Demand', 'Fulfilled_Demand', 'Unfulfilled_Demand', 'Wastage_Units',
           'Shortage_Units', 'Wastage_Rate_Pct', 'Shortage_Rate_Pct', 'Wastage_Cost',
           'Shortage_Cost', 'Holding_Cost', 'Order_Cost', 'Total_Cost']
H2_COLS = H1_COLS
SC_COLS = ['Total_Wastage', 'Total_Shortage', 'Trans_H1_to_H2', 'Trans_H2_to_H1',
           'Trans_Cost_H1_to_H2', 'Trans_Cost_H2_to_H1', 'Network_Total_Cost']


def compute_episode_summary(df, cols):
    s = {}
    for c in cols:
        if 'Cost' in c or 'Units' in c or 'Demand' in c or 'Trans' in c:
            s[c] = df[c].sum()
        else:
            s[c] = df[c].mean()
    return s


def aggregate_seed_summaries(seed_summaries):
    df = pd.DataFrame(seed_summaries)
    result = {}
    for c in df.columns:
        result[f'{c}_Mean'] = df[c].mean()
        result[f'{c}_Std'] = df[c].std()
    return pd.DataFrame([result])


def evaluate_multi_seed(model_path, train_seed, eval_seeds=range(1, 51)):
    print(f"\n{'='*80}")
    print(f"🚀 EVALUATING MODEL (Train Seed {train_seed}) ON {len(eval_seeds)} SEEDS")
    print(f"{'='*80}")

    env = PlateletSupplyChainEnv()
    ppo = SingleAgentPPO()
    ppo.load(model_path)
    ppo.actor.eval()

    h1_records_all = []
    h2_records_all = []
    sc_records_all = []

    for seed in tqdm(eval_seeds, desc=f"Eval Train-Seed {train_seed}"):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        state = env.reset()
        h1_daily = []
        h2_daily = []
        sc_daily = []

        for day in range(env.episode_length):
            action = ppo.get_deterministic_action(state)
            next_state, reward, done, info = env.step(action)

            d1 = info['demand_h1']
            s1 = info['shortage_h1']
            w1 = info['wastage_h1']
            fulfilled1 = d1 - s1
            inv_h1_before = info['inv_h1_before']
            inv_h2_before = info['inv_h2_before']
            total_inv_before = inv_h1_before + inv_h2_before

            h1_hold = (inv_h1_before / total_inv_before) * info['inventory_holding_cost'] if total_inv_before > 0 else 0.0

            h1_daily.append({
                'Day': day + 1,
                'Total_Demand': d1,
                'Fulfilled_Demand': fulfilled1,
                'Unfulfilled_Demand': s1,
                'Wastage_Units': w1,
                'Shortage_Units': s1,
                'Wastage_Rate_Pct': (w1 / d1 * 100) if d1 > 0 else 0.0,
                'Shortage_Rate_Pct': (s1 / d1 * 100) if d1 > 0 else 0.0,
                'Wastage_Cost': w1 * env.wastage_cost,
                'Shortage_Cost': s1 * env.shortage_cost,
                'Holding_Cost': h1_hold,
                'Order_Cost': info['actual_order_h1'] * env.order_cost,
                'Total_Cost': (w1 * env.wastage_cost) + (s1 * env.shortage_cost) + h1_hold + (info['actual_order_h1'] * env.order_cost),
            })

            d2 = info['demand_h2']
            s2 = info['shortage_h2']
            w2 = info['wastage_h2']
            fulfilled2 = d2 - s2

            h2_hold = (inv_h2_before / total_inv_before) * info['inventory_holding_cost'] if total_inv_before > 0 else 0.0

            h2_daily.append({
                'Day': day + 1,
                'Total_Demand': d2,
                'Fulfilled_Demand': fulfilled2,
                'Unfulfilled_Demand': s2,
                'Wastage_Units': w2,
                'Shortage_Units': s2,
                'Wastage_Rate_Pct': (w2 / d2 * 100) if d2 > 0 else 0.0,
                'Shortage_Rate_Pct': (s2 / d2 * 100) if d2 > 0 else 0.0,
                'Wastage_Cost': w2 * env.wastage_cost,
                'Shortage_Cost': s2 * env.shortage_cost,
                'Holding_Cost': h2_hold,
                'Order_Cost': info['actual_order_h2'] * env.order_cost,
                'Total_Cost': (w2 * env.wastage_cost) + (s2 * env.shortage_cost) + h2_hold + (info['actual_order_h2'] * env.order_cost),
            })

            total_trans = info['transferred_h1_to_h2'] + info['transferred_h2_to_h1']
            trans_cost_total = info['transshipment_cost']

            if total_trans > 0:
                trans_cost_h1_to_h2 = (info['transferred_h1_to_h2'] / total_trans) * trans_cost_total
                trans_cost_h2_to_h1 = (info['transferred_h2_to_h1'] / total_trans) * trans_cost_total
            else:
                trans_cost_h1_to_h2 = 0.0
                trans_cost_h2_to_h1 = 0.0

            sc_daily.append({
                'Day': day + 1,
                'Total_Wastage': w1 + w2,
                'Total_Shortage': s1 + s2,
                'Trans_H1_to_H2': info['transferred_h1_to_h2'],
                'Trans_H2_to_H1': info['transferred_h2_to_h1'],
                'Trans_Cost_H1_to_H2': trans_cost_h1_to_h2,
                'Trans_Cost_H2_to_H1': trans_cost_h2_to_h1,
                'Network_Total_Cost': info['total_cost'],
            })

            state = next_state
            if done:
                break

        h1_records_all.append(pd.DataFrame(h1_daily))
        h2_records_all.append(pd.DataFrame(h2_daily))
        sc_records_all.append(pd.DataFrame(sc_daily))

    h1_seed_summaries = [compute_episode_summary(df, H1_COLS) for df in h1_records_all]
    h2_seed_summaries = [compute_episode_summary(df, H2_COLS) for df in h2_records_all]
    sc_seed_summaries = [compute_episode_summary(df, SC_COLS) for df in sc_records_all]

    h1_model_summary = aggregate_seed_summaries(h1_seed_summaries)
    h2_model_summary = aggregate_seed_summaries(h2_seed_summaries)
    sc_model_summary = aggregate_seed_summaries(sc_seed_summaries)

    raw_rows = []
    for i, seed in enumerate(eval_seeds):
        row = {'Train_Seed': train_seed, 'Eval_Seed': seed}
        for k, v in h1_seed_summaries[i].items():
            row[f'H1_{k}'] = v
        for k, v in h2_seed_summaries[i].items():
            row[f'H2_{k}'] = v
        for k, v in sc_seed_summaries[i].items():
            row[f'SC_{k}'] = v
        raw_rows.append(row)

    return h1_model_summary, h2_model_summary, sc_model_summary, raw_rows


# =============================================================================
# EXCEL BUILDER
# =============================================================================
def create_excel(all_h1_summaries, all_h2_summaries, all_sc_summaries, raw_rows, train_seeds):
    excel_path = f"{OUTPUT_PREFIX}_ALL_RESULTS.xlsx"
    print(f"\n{'='*80}")
    print(f"📊 CREATING EXCEL: {excel_path}")
    print(f"{'='*80}")

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

        def write_summary_sheet(dfs, sheet_name):
            combined = pd.concat(dfs, ignore_index=True)
            mean_cols = [c for c in combined.columns if c.endswith('_Mean')]
            result = pd.DataFrame({
                'Metric': [c.replace('_Mean', '') for c in mean_cols],
                'Mean': [combined[c].mean() for c in mean_cols],
                'Std': [combined[c].std() for c in mean_cols],
                'Min': [combined[c].min() for c in mean_cols],
                'Max': [combined[c].max() for c in mean_cols]
            })
            result.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"💾 Sheet '{sheet_name}': {len(result)} metrics")
            return result

        h1_summary_df = write_summary_sheet(all_h1_summaries, 'H1_Summary')
        h2_summary_df = write_summary_sheet(all_h2_summaries, 'H2_Summary')
        sc_summary_df = write_summary_sheet(all_sc_summaries, 'SC_Summary')

        # Master
        master_rows = []
        for _, row in h1_summary_df.iterrows():
            master_rows.append({'Table': 'H1', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        for _, row in h2_summary_df.iterrows():
            master_rows.append({'Table': 'H2', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        for _, row in sc_summary_df.iterrows():
            master_rows.append({'Table': 'SC', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        master_df = pd.DataFrame(master_rows)
        master_df.to_excel(writer, sheet_name='Master', index=False)
        print(f"💾 Sheet 'Master': {len(master_df)} rows")

        # Model_Comparison
        comparison_rows = []
        for seed, h1, h2, sc in zip(train_seeds, all_h1_summaries, all_h2_summaries, all_sc_summaries):
            comparison_rows.append({
                'Train_Seed': seed,
                'H1_Total_Cost': h1['Total_Cost_Mean'].values[0] if 'Total_Cost_Mean' in h1.columns else np.nan,
                'H2_Total_Cost': h2['Total_Cost_Mean'].values[0] if 'Total_Cost_Mean' in h2.columns else np.nan,
                'Network_Total_Cost': sc['Network_Total_Cost_Mean'].values[0] if 'Network_Total_Cost_Mean' in sc.columns else np.nan,
                'H1_Wastage_Rate': h1['Wastage_Rate_Pct_Mean'].values[0] if 'Wastage_Rate_Pct_Mean' in h1.columns else np.nan,
                'H2_Wastage_Rate': h2['Wastage_Rate_Pct_Mean'].values[0] if 'Wastage_Rate_Pct_Mean' in h2.columns else np.nan,
                'H1_Shortage_Rate': h1['Shortage_Rate_Pct_Mean'].values[0] if 'Shortage_Rate_Pct_Mean' in h1.columns else np.nan,
                'H2_Shortage_Rate': h2['Shortage_Rate_Pct_Mean'].values[0] if 'Shortage_Rate_Pct_Mean' in h2.columns else np.nan,
            })
        comparison_df = pd.DataFrame(comparison_rows)
        comparison_df.to_excel(writer, sheet_name='Model_Comparison', index=False)
        print(f"💾 Sheet 'Model_Comparison': {len(comparison_df)} models")

        # Raw_Episode_Data
        raw_df = pd.DataFrame(raw_rows)
        raw_df.to_excel(writer, sheet_name='Raw_Episode_Data', index=False)
        print(f"💾 Sheet 'Raw_Episode_Data': {len(raw_df)} rows")

    print(f"\n✅ Excel saved: {excel_path}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    TRAIN_SEEDS = [42, 123, 456, 789, 2025]

    all_rewards = {}
    trained_models = []

    # 1) Train 5 models
    for seed in TRAIN_SEEDS:
        path, rewards = train(train_seed=seed)
        trained_models.append((seed, path))
        all_rewards[seed] = rewards

    # 2) Combined learning curve (only this PNG)
    plot_combined_learning_curves(all_rewards)

    # 3) Evaluate each model on 50 seeds
    all_h1_summaries = []
    all_h2_summaries = []
    all_sc_summaries = []
    all_raw_rows = []

    for seed, path in trained_models:
        h1_mod, h2_mod, sc_mod, raw = evaluate_multi_seed(path, train_seed=seed, eval_seeds=range(1, 51))
        all_h1_summaries.append(h1_mod)
        all_h2_summaries.append(h2_mod)
        all_sc_summaries.append(sc_mod)
        all_raw_rows.extend(raw)

    # 4) One Excel file with 6 sheets
    create_excel(all_h1_summaries, all_h2_summaries, all_sc_summaries, all_raw_rows, TRAIN_SEEDS)


🚀 TRAINING SEED 42 — Single-Agent PPO (No Transshipment)


Train Seed 42: 100%|██████████| 2000/2000 [02:43<00:00, 12.24it/s]



🚀 TRAINING SEED 123 — Single-Agent PPO (No Transshipment)


Train Seed 123: 100%|██████████| 2000/2000 [02:43<00:00, 12.20it/s]



🚀 TRAINING SEED 456 — Single-Agent PPO (No Transshipment)


...


🚀 EVALUATING MODEL (Train Seed 789) ON 50 SEEDS


Eval Train-Seed 789: 100%|██████████| 50/50 [00:02<00:00, 21.13it/s]



🚀 EVALUATING MODEL (Train Seed 2025) ON 50 SEEDS


Eval Train-Seed 2025: 100%|██████████| 50/50 [00:03<00:00, 15.30it/s]



📊 CREATING EXCEL: single_agent_without_transshipment_ALL_RESULTS.xlsx
💾 Sheet 'H1_Summary': 12 metrics
💾 Sheet 'H2_Summary': 12 metrics
💾 Sheet 'SC_Summary': 7 metrics
💾 Sheet 'Master': 31 rows
💾 Sheet 'Model_Comparison': 5 models
💾 Sheet 'Raw_Episode_Data': 250 rows

✅ Excel saved: single_agent_without_transshipment_ALL_RESULTS.xlsx


# Order-Up-To Baseline

In [4]:
# -*- coding: utf-8 -*-
"""Order-Up-To Baseline — Single Cell for Google Colab"""
import random
import numpy as np
import pandas as pd
import torch
from collections import deque
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION & HYPERPARAMETERS
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ZINB_H1_PARAMS = [{"phi": 0.25, "p": 0.48, "n": 15}]
ZINB_H2_PARAMS = [{"phi": 0.25, "p": 0.48, "n": 15}]

def _zinb_mean(params: dict) -> float:
    phi, p, n = params["phi"], params["p"], params["n"]
    return (1.0 - phi) * n * (1.0 - p) / p

def sample_zinb(phi: float, p: float, n: int) -> int:
    if np.random.random() < phi:
        return 0
    return int(np.random.negative_binomial(n, p))

def sample_hospital_demand(zinb_params: list, mult: float = 1.0) -> int:
    total = sum(sample_zinb(p["phi"], p["p"], p["n"]) for p in zinb_params)
    return max(0, int(total * mult))

ENV_CONFIG = {
    "max_capacity": 100,
    "shelf_life": 5,
    "order_cost": 1.0,
    "transshipment_cost": 1.0,
    "transshipment_fixed_cost": 5.0,
    "shortage_cost": 25.0,
    "wastage_cost": 15.0,
    "inventory_cost": 0.5,
    "episode_length": 60,
    "order_cap": 70,
    "trans_cap": 50,
    "transshipment_deadband": 0.1,
    "disruption_threshold": 1.2,
    "disruption_prob": 0.02,
    "disruption_min_len": 3,
    "disruption_max_len": 8,
    "disruption_mult": 1.4,
}

ORDER_SCHEDULE_H1 = {1, 4, 6}
ORDER_SCHEDULE_H2 = {0, 3, 5}

STATE_DIM = 26
REWARD_SCALE = 100.0

OUTPUT_PREFIX = "order_upto_baseline"

# =============================================================================
# ENVIRONMENT (NO TRANSSHIPMENT — aligned with MAPPO)
# =============================================================================
class PlateletSupplyChainEnv:
    def __init__(self):
        for k, v in ENV_CONFIG.items():
            setattr(self, k, v)
        self.num_hospitals = 2
        self.demand_mean_h1 = sum(_zinb_mean(p) for p in ZINB_H1_PARAMS)
        self.demand_mean_h2 = sum(_zinb_mean(p) for p in ZINB_H2_PARAMS)
        self.reset()

    def _is_order_day_h1(self):
        return (self.day % 7) in ORDER_SCHEDULE_H1

    def _is_order_day_h2(self):
        return (self.day % 7) in ORDER_SCHEDULE_H2

    def _compute_disruption_flag(self, history):
        if len(history) < 5:
            return 0.0
        ma2 = np.mean(list(history)[-2:])
        ma30 = np.mean(list(history)[-30:])
        return 1.0 if (ma30 > 0 and ma2 > self.disruption_threshold * ma30) else 0.0

    def _get_observation(self):
        obs = np.zeros((STATE_DIM,), dtype=np.float32)
        idx = 0
        for h in range(self.num_hospitals):
            start = h * self.shelf_life
            for age in range(self.shelf_life):
                obs[idx] = self.state[start + age]
                idx += 1

        inv_h1 = np.sum(self.state[0:self.shelf_life])
        inv_h2 = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        obs[10] = self.day / self.episode_length
        obs[11] = self._compute_disruption_flag(self.demand_history_h1)
        obs[12] = self._compute_disruption_flag(self.demand_history_h2)
        obs[13] = inv_h1 / self.max_capacity
        obs[14] = inv_h2 / self.max_capacity
        obs[15] = 1.0 if self._is_order_day_h1() else 0.0
        obs[16] = 1.0 if self._is_order_day_h2() else 0.0
        obs[17] = (inv_h1 - inv_h2) / self.max_capacity
        obs[18] = self.crisis_counter_h1 / self.disruption_max_len
        obs[19] = self.crisis_counter_h2 / self.disruption_max_len

        d1 = np.mean(self.demand_history_h1[-7:]) if self.demand_history_h1 else self.demand_mean_h1
        d2 = np.mean(self.demand_history_h2[-7:]) if self.demand_history_h2 else self.demand_mean_h2
        obs[20] = min(inv_h1 / (d1 * 7 + 1e-8), 3.0)
        obs[21] = min(inv_h2 / (d2 * 7 + 1e-8), 3.0)

        if self.demand_history_h1:
            td1 = sum(self.demand_history_h1[-30:])
            obs[22] = sum(self.shortage_history_h1) / (td1 + 1e-8)
            obs[24] = sum(self.wastage_history_h1) / (td1 + 1e-8)
        if self.demand_history_h2:
            td2 = sum(self.demand_history_h2[-30:])
            obs[23] = sum(self.shortage_history_h2) / (td2 + 1e-8)
            obs[25] = sum(self.wastage_history_h2) / (td2 + 1e-8)
        return obs.copy()

    def reset(self):
        self.state = np.zeros((self.num_hospitals * self.shelf_life,), dtype=np.float32)
        self.day = 0
        self.demand_history_h1, self.demand_history_h2 = [], []
        self.shortage_history_h1, self.shortage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.wastage_history_h1, self.wastage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.disruption_counter_h1 = self.disruption_counter_h2 = 0
        self.crisis_counter_h1 = self.crisis_counter_h2 = 0
        return self._get_observation()

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)

        order_h1 = action[0] * self.order_cap if self._is_order_day_h1() else 0.0
        order_h2 = action[1] * self.order_cap if self._is_order_day_h2() else 0.0

        inv_h1_before = np.sum(self.state[0:self.shelf_life])
        inv_h2_before = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        actual_order_h1 = self._process_order(0, min(order_h1, max(0.0, self.max_capacity - inv_h1_before)))
        actual_order_h2 = self._process_order(1, min(order_h2, max(0.0, self.max_capacity - inv_h2_before)))

        # No transshipment
        actual_trans_h1_to_h2 = 0.0
        actual_trans_h2_to_h1 = 0.0

        if self.day >= 5:
            if self.disruption_counter_h1 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h1 = random.randint(self.disruption_min_len, self.disruption_max_len)
            if self.disruption_counter_h2 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h2 = random.randint(self.disruption_min_len, self.disruption_max_len)

        mult_h1 = self.disruption_mult if self.disruption_counter_h1 > 0 else 1.0
        mult_h2 = self.disruption_mult if self.disruption_counter_h2 > 0 else 1.0

        if self.disruption_counter_h1 > 0:
            self.disruption_counter_h1 -= 1
        if self.disruption_counter_h2 > 0:
            self.disruption_counter_h2 -= 1

        demand_h1 = sample_hospital_demand(ZINB_H1_PARAMS, mult=mult_h1)
        demand_h2 = sample_hospital_demand(ZINB_H2_PARAMS, mult=mult_h2)

        self.demand_history_h1.append(demand_h1)
        self.demand_history_h2.append(demand_h2)

        self.crisis_counter_h1 = min(self.crisis_counter_h1 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h1) > 0.5 else 0
        self.crisis_counter_h2 = min(self.crisis_counter_h2 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h2) > 0.5 else 0

        wastage_h1, shortage_h1, _ = self._fulfill_demand(0, demand_h1)
        wastage_h2, shortage_h2, _ = self._fulfill_demand(1, demand_h2)

        self.shortage_history_h1.append(shortage_h1)
        self.shortage_history_h2.append(shortage_h2)
        self.wastage_history_h1.append(wastage_h1)
        self.wastage_history_h2.append(wastage_h2)

        self._age_blood()

        inv_h1_after = np.sum(self.state[0:self.shelf_life])
        inv_h2_after = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        inventory_holding_cost = (inv_h1_before + inv_h2_before) * self.inventory_cost
        self.day += 1

        transshipment_cost = 0.0
        wastage_cost = (wastage_h1 + wastage_h2) * self.wastage_cost
        shortage_cost = (shortage_h1 + shortage_h2) * self.shortage_cost
        order_cost = (actual_order_h1 + actual_order_h2) * self.order_cost

        total_cost = wastage_cost + shortage_cost + order_cost + inventory_holding_cost
        shared_reward = -total_cost / REWARD_SCALE

        info = {
            'order_mask': [1.0 if self._is_order_day_h1() else 0.0, 1.0 if self._is_order_day_h2() else 0.0],
            'demand_h1': demand_h1, 'demand_h2': demand_h2,
            'inventory_h1': inv_h1_after, 'inventory_h2': inv_h2_after,
            'order_h1': order_h1, 'order_h2': order_h2,
            'actual_order_h1': actual_order_h1, 'actual_order_h2': actual_order_h2,
            'transferred_h1_to_h2': actual_trans_h1_to_h2, 'transferred_h2_to_h1': actual_trans_h2_to_h1,
            'shortage_h1': shortage_h1, 'shortage_h2': shortage_h2,
            'wastage_h1': wastage_h1, 'wastage_h2': wastage_h2,
            'total_cost': total_cost, 'wastage_cost': wastage_cost,
            'shortage_cost': shortage_cost, 'transshipment_cost': transshipment_cost,
            'order_cost': order_cost, 'inventory_holding_cost': inventory_holding_cost,
            'disruption_active_h1': mult_h1 > 1.0, 'disruption_active_h2': mult_h2 > 1.0,
            'crisis_counter_h1': self.crisis_counter_h1, 'crisis_counter_h2': self.crisis_counter_h2,
            'inv_h1_before': inv_h1_before, 'inv_h2_before': inv_h2_before,
        }

        done = self.day >= self.episode_length
        return self._get_observation(), shared_reward, done, info

    def _process_order(self, hospital, amount):
        if amount <= 0:
            return 0.0
        start = hospital * self.shelf_life
        accepted = max(0.0, min(amount, self.max_capacity - np.sum(self.state[start:start + self.shelf_life])))
        self.state[start] += accepted
        return accepted

    def _fulfill_demand(self, hospital, demand):
        s_idx, e_idx = hospital * self.shelf_life, hospital * self.shelf_life + self.shelf_life
        unfulfilled = demand
        for i in range(e_idx - 1, s_idx - 1, -1):
            if self.state[i] > 0 and unfulfilled > 0:
                used = min(self.state[i], unfulfilled)
                self.state[i] -= used
                unfulfilled -= used
            if unfulfilled <= 1e-5:
                break
        wastage = self.state[e_idx - 1]
        self.state[e_idx - 1] = 0
        return wastage, unfulfilled, demand - unfulfilled

    def _age_blood(self):
        for h in range(self.num_hospitals):
            s_idx, e_idx = h * self.shelf_life, h * self.shelf_life + self.shelf_life
            self.state[s_idx + 1:e_idx] = self.state[s_idx:e_idx - 1]
            self.state[s_idx] = 0


# =============================================================================
# BASELINE: ORDER-UP-TO (NO TRANSSHIPMENT)
# =============================================================================
class OrderUpToBaseline:
    """
    سیاست Order-Up-To (Base-Stock) بدون انتقال جانبی.
    S1 = S2 = percentage * max_capacity
    """
    def __init__(self, S1=120, S2=120):
        self.S1 = S1
        self.S2 = S2
        self.order_cap = ENV_CONFIG["order_cap"]

    def get_action(self, env: PlateletSupplyChainEnv, obs: np.ndarray):
        inv_h1 = np.sum(env.state[0:env.shelf_life])
        inv_h2 = np.sum(env.state[env.shelf_life:env.shelf_life*2])

        # Procurement (Order-Up-To)
        if env._is_order_day_h1():
            order_h1 = max(0.0, self.S1 - inv_h1)
            order_h1 = min(order_h1, self.order_cap)
        else:
            order_h1 = 0.0

        if env._is_order_day_h2():
            order_h2 = max(0.0, self.S2 - inv_h2)
            order_h2 = min(order_h2, self.order_cap)
        else:
            order_h2 = 0.0

        # NO Transshipment — neutral action for compatibility
        action = np.array([
            order_h1 / self.order_cap,
            order_h2 / self.order_cap,
            0.5  # neutral transshipment action (no transfer)
        ], dtype=np.float32)

        return np.clip(action, 0.0, 1.0)


# =============================================================================
# EVALUATION HELPERS (Same structure as RL models)
# =============================================================================
H1_COLS = ['Total_Demand', 'Fulfilled_Demand', 'Unfulfilled_Demand', 'Wastage_Units',
           'Shortage_Units', 'Wastage_Rate_Pct', 'Shortage_Rate_Pct', 'Wastage_Cost',
           'Shortage_Cost', 'Holding_Cost', 'Order_Cost', 'Total_Cost']
H2_COLS = H1_COLS
SC_COLS = ['Total_Wastage', 'Total_Shortage', 'Trans_H1_to_H2', 'Trans_H2_to_H1',
           'Trans_Cost_H1_to_H2', 'Trans_Cost_H2_to_H1', 'Network_Total_Cost']


def compute_episode_summary(df, cols):
    s = {}
    for c in cols:
        if 'Cost' in c or 'Units' in c or 'Demand' in c or 'Trans' in c:
            s[c] = df[c].sum()
        else:
            s[c] = df[c].mean()
    return s


def aggregate_seed_summaries(seed_summaries):
    df = pd.DataFrame(seed_summaries)
    result = {}
    for c in df.columns:
        result[f'{c}_Mean'] = df[c].mean()
        result[f'{c}_Std'] = df[c].std()
    return pd.DataFrame([result])


# =============================================================================
# MULTI-SEED EVALUATION — Each policy treated as a "model"
# =============================================================================
def evaluate_all_baselines(eval_seeds=range(1, 51)):
    max_cap = ENV_CONFIG["max_capacity"]
    percentages = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 98, 99]
    S_levels = [int(max_cap * p / 100) for p in percentages]

    print(f"\n{'='*90}")
    print(f"🚀 BASELINE COMPARISON: Order-Up-To at {len(percentages)} Capacity Levels")
    print(f"   Max Capacity = {max_cap}, S levels = {S_levels}")
    print(f"   Eval Seeds = {len(eval_seeds)} ({eval_seeds.start} to {eval_seeds.stop-1})")
    print(f"{'='*90}")

    env = PlateletSupplyChainEnv()

    # Treat each policy as a "model" — exactly like RL models
    all_h1_summaries = []
    all_h2_summaries = []
    all_sc_summaries = []
    all_raw_rows = []

    for pct, S in zip(percentages, S_levels):
        policy_name = f"S{pct}"
        print(f"\n📊 Evaluating Policy: {policy_name} (S1=S2={S})...")

        agent = OrderUpToBaseline(S1=S, S2=S)

        h1_records_all = []
        h2_records_all = []
        sc_records_all = []

        for seed in tqdm(eval_seeds, desc=f"  {policy_name}", leave=False):
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)

            state = env.reset()
            h1_daily = []
            h2_daily = []
            sc_daily = []

            for day in range(env.episode_length):
                inv_h1_before = np.sum(env.state[0:env.shelf_life])
                inv_h2_before = np.sum(env.state[env.shelf_life:env.shelf_life*2])
                total_inv_before = inv_h1_before + inv_h2_before

                action = agent.get_action(env, state)
                next_state, reward, done, info = env.step(action)

                # --- Hospital 1 ---
                d1 = info['demand_h1']
                s1 = info['shortage_h1']
                w1 = info['wastage_h1']
                fulfilled1 = d1 - s1

                h1_hold = (inv_h1_before / total_inv_before) * info['inventory_holding_cost'] if total_inv_before > 0 else 0.0

                h1_daily.append({
                    'Day': day + 1,
                    'Total_Demand': d1,
                    'Fulfilled_Demand': fulfilled1,
                    'Unfulfilled_Demand': s1,
                    'Wastage_Units': w1,
                    'Shortage_Units': s1,
                    'Wastage_Rate_Pct': (w1 / d1 * 100) if d1 > 0 else 0.0,
                    'Shortage_Rate_Pct': (s1 / d1 * 100) if d1 > 0 else 0.0,
                    'Wastage_Cost': w1 * env.wastage_cost,
                    'Shortage_Cost': s1 * env.shortage_cost,
                    'Holding_Cost': h1_hold,
                    'Order_Cost': info['actual_order_h1'] * env.order_cost,
                    'Total_Cost': (w1 * env.wastage_cost) + (s1 * env.shortage_cost) + h1_hold + (info['actual_order_h1'] * env.order_cost),
                })

                # --- Hospital 2 ---
                d2 = info['demand_h2']
                s2 = info['shortage_h2']
                w2 = info['wastage_h2']
                fulfilled2 = d2 - s2

                h2_hold = (inv_h2_before / total_inv_before) * info['inventory_holding_cost'] if total_inv_before > 0 else 0.0

                h2_daily.append({
                    'Day': day + 1,
                    'Total_Demand': d2,
                    'Fulfilled_Demand': fulfilled2,
                    'Unfulfilled_Demand': s2,
                    'Wastage_Units': w2,
                    'Shortage_Units': s2,
                    'Wastage_Rate_Pct': (w2 / d2 * 100) if d2 > 0 else 0.0,
                    'Shortage_Rate_Pct': (s2 / d2 * 100) if d2 > 0 else 0.0,
                    'Wastage_Cost': w2 * env.wastage_cost,
                    'Shortage_Cost': s2 * env.shortage_cost,
                    'Holding_Cost': h2_hold,
                    'Order_Cost': info['actual_order_h2'] * env.order_cost,
                    'Total_Cost': (w2 * env.wastage_cost) + (s2 * env.shortage_cost) + h2_hold + (info['actual_order_h2'] * env.order_cost),
                })

                # --- Supply Chain ---
                total_trans = info['transferred_h1_to_h2'] + info['transferred_h2_to_h1']
                trans_cost_total = info['transshipment_cost']

                if total_trans > 0:
                    trans_cost_h1_to_h2 = (info['transferred_h1_to_h2'] / total_trans) * trans_cost_total
                    trans_cost_h2_to_h1 = (info['transferred_h2_to_h1'] / total_trans) * trans_cost_total
                else:
                    trans_cost_h1_to_h2 = 0.0
                    trans_cost_h2_to_h1 = 0.0

                sc_daily.append({
                    'Day': day + 1,
                    'Total_Wastage': w1 + w2,
                    'Total_Shortage': s1 + s2,
                    'Trans_H1_to_H2': info['transferred_h1_to_h2'],
                    'Trans_H2_to_H1': info['transferred_h2_to_h1'],
                    'Trans_Cost_H1_to_H2': trans_cost_h1_to_h2,
                    'Trans_Cost_H2_to_H1': trans_cost_h2_to_h1,
                    'Network_Total_Cost': info['total_cost'],
                })

                state = next_state
                if done:
                    break

            h1_records_all.append(pd.DataFrame(h1_daily))
            h2_records_all.append(pd.DataFrame(h2_daily))
            sc_records_all.append(pd.DataFrame(sc_daily))

        # Aggregate this "model" (policy) across 50 seeds
        h1_seed_summaries = [compute_episode_summary(df, H1_COLS) for df in h1_records_all]
        h2_seed_summaries = [compute_episode_summary(df, H2_COLS) for df in h2_records_all]
        sc_seed_summaries = [compute_episode_summary(df, SC_COLS) for df in sc_records_all]

        h1_model_summary = aggregate_seed_summaries(h1_seed_summaries)
        h2_model_summary = aggregate_seed_summaries(h2_seed_summaries)
        sc_model_summary = aggregate_seed_summaries(sc_seed_summaries)

        all_h1_summaries.append(h1_model_summary)
        all_h2_summaries.append(h2_model_summary)
        all_sc_summaries.append(sc_model_summary)

        # Raw rows for this "model"
        for i, seed in enumerate(eval_seeds):
            row = {'Train_Seed': policy_name, 'Eval_Seed': seed}
            for k, v in h1_seed_summaries[i].items():
                row[f'H1_{k}'] = v
            for k, v in h2_seed_summaries[i].items():
                row[f'H2_{k}'] = v
            for k, v in sc_seed_summaries[i].items():
                row[f'SC_{k}'] = v
            all_raw_rows.append(row)

    return all_h1_summaries, all_h2_summaries, all_sc_summaries, all_raw_rows, percentages


# =============================================================================
# EXCEL BUILDER (Same structure as RL models)
# =============================================================================
def create_excel(all_h1_summaries, all_h2_summaries, all_sc_summaries, raw_rows, policy_names):
    excel_path = f"{OUTPUT_PREFIX}_ALL_RESULTS.xlsx"
    print(f"\n{'='*80}")
    print(f"📊 CREATING EXCEL: {excel_path}")
    print(f"{'='*80}")

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

        def write_summary_sheet(dfs, sheet_name):
            combined = pd.concat(dfs, ignore_index=True)
            mean_cols = [c for c in combined.columns if c.endswith('_Mean')]
            result = pd.DataFrame({
                'Metric': [c.replace('_Mean', '') for c in mean_cols],
                'Mean': [combined[c].mean() for c in mean_cols],
                'Std': [combined[c].std() for c in mean_cols],
                'Min': [combined[c].min() for c in mean_cols],
                'Max': [combined[c].max() for c in mean_cols]
            })
            result.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"💾 Sheet '{sheet_name}': {len(result)} metrics")
            return result

        h1_summary_df = write_summary_sheet(all_h1_summaries, 'H1_Summary')
        h2_summary_df = write_summary_sheet(all_h2_summaries, 'H2_Summary')
        sc_summary_df = write_summary_sheet(all_sc_summaries, 'SC_Summary')

        # Master
        master_rows = []
        for _, row in h1_summary_df.iterrows():
            master_rows.append({'Table': 'H1', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        for _, row in h2_summary_df.iterrows():
            master_rows.append({'Table': 'H2', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        for _, row in sc_summary_df.iterrows():
            master_rows.append({'Table': 'SC', 'Metric': row['Metric'], 'Mean': row['Mean'],
                                'Std': row['Std'], 'Min': row['Min'], 'Max': row['Max']})
        master_df = pd.DataFrame(master_rows)
        master_df.to_excel(writer, sheet_name='Master', index=False)
        print(f"💾 Sheet 'Master': {len(master_df)} rows")

        # Model_Comparison — each policy is a "model"
        comparison_rows = []
        for pname, h1, h2, sc in zip(policy_names, all_h1_summaries, all_h2_summaries, all_sc_summaries):
            comparison_rows.append({
                'Train_Seed': f"S{pname}",
                'H1_Total_Cost': h1['Total_Cost_Mean'].values[0] if 'Total_Cost_Mean' in h1.columns else np.nan,
                'H2_Total_Cost': h2['Total_Cost_Mean'].values[0] if 'Total_Cost_Mean' in h2.columns else np.nan,
                'Network_Total_Cost': sc['Network_Total_Cost_Mean'].values[0] if 'Network_Total_Cost_Mean' in sc.columns else np.nan,
                'H1_Wastage_Rate': h1['Wastage_Rate_Pct_Mean'].values[0] if 'Wastage_Rate_Pct_Mean' in h1.columns else np.nan,
                'H2_Wastage_Rate': h2['Wastage_Rate_Pct_Mean'].values[0] if 'Wastage_Rate_Pct_Mean' in h2.columns else np.nan,
                'H1_Shortage_Rate': h1['Shortage_Rate_Pct_Mean'].values[0] if 'Shortage_Rate_Pct_Mean' in h1.columns else np.nan,
                'H2_Shortage_Rate': h2['Shortage_Rate_Pct_Mean'].values[0] if 'Shortage_Rate_Pct_Mean' in h2.columns else np.nan,
            })
        comparison_df = pd.DataFrame(comparison_rows)
        comparison_df.to_excel(writer, sheet_name='Model_Comparison', index=False)
        print(f"💾 Sheet 'Model_Comparison': {len(comparison_df)} policies")

        # Raw_Episode_Data
        raw_df = pd.DataFrame(raw_rows)
        raw_df.to_excel(writer, sheet_name='Raw_Episode_Data', index=False)
        print(f"💾 Sheet 'Raw_Episode_Data': {len(raw_df)} rows")

    print(f"\n✅ Excel saved: {excel_path}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    # Evaluate all baseline policies (each treated as a "model")
    all_h1, all_h2, all_sc, raw_rows, percentages = evaluate_all_baselines(eval_seeds=range(1, 51))

    # Create Excel with same structure as RL models
    policy_names = [f"{p}" for p in percentages]  # e.g., "10", "20", ...
    create_excel(all_h1, all_h2, all_sc, raw_rows, policy_names)


🚀 BASELINE COMPARISON: Order-Up-To at 12 Capacity Levels
   Max Capacity = 100, S levels = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 98, 99]
   Eval Seeds = 50 (1 to 50)

📊 Evaluating Policy: S10 (S1=S2=10)...



📊 Evaluating Policy: S20 (S1=S2=20)...



📊 Evaluating Policy: S30 (S1=S2=30)...



📊 Evaluating Policy: S40 (S1=S2=40)...



📊 Evaluating Policy: S50 (S1=S2=50)...


...


📊 Evaluating Policy: S90 (S1=S2=90)...



📊 Evaluating Policy: S95 (S1=S2=95)...



📊 Evaluating Policy: S98 (S1=S2=98)...



📊 Evaluating Policy: S99 (S1=S2=99)...



📊 CREATING EXCEL: order_upto_baseline_ALL_RESULTS.xlsx
💾 Sheet 'H1_Summary': 12 metrics
💾 Sheet 'H2_Summary': 12 metrics
💾 Sheet 'SC_Summary': 7 metrics
💾 Sheet 'Master': 31 rows
💾 Sheet 'Model_Comparison': 12 policies
💾 Sheet 'Raw_Episode_Data': 600 rows

✅ Excel saved: order_upto_baseline_ALL_RESULTS.xlsx
